In [1]:
# CHANGE KERNEL TO SAGE MATH 10.2 FIRST 

# impact point of muon
phi0,rho=var('phi0,rho', domain='real')  
# inclination of muon (same notation as in Ahnen et al., 2019)
nu,psi=var('nu,psi', domain='real')
# photon emission parameters
theta_c,phi,m,L=var('theta_c,phi,m,L', domain='real')  
# mirror parameters 
x,y,c=var('x,y,c', domain='real') 
# helper for some Taylor expansions
eps=var('eps',domain='real')

In [2]:
# Emission point of the photon: we move to the impact point I and then L times up the muon vector M to emission point O  

# Note the convention on the impact angle phi0 as being that with the largest cord, hence: 
# phi0 --> phi0 + 180 deg 
# See, e.g., fig. 1 of Ahnen et al. 2019

order_expansion = 2   # default order of Taylor expansion for nu, theta_c and c

II = vector([rho*cos(phi0+pi),rho*sin(phi0+pi),c*rho^2])    # muon impact point on a parabolic mirror, see Eq. (8), cannot be I because of confusion with sqrt(-1)
M = vector([sin(nu)*cos(psi),sin(nu)*sin(psi),-cos(nu)])    # muon vector, normalized to 1, pointing downwards, see Eq. (1) 
E = II-L*M                                                  # photon emission point, see Eqs. (9) and (10)

print ('E=',E)
E = vector([E[0].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full(),
            E[1].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full(),
            E[2].collect(nu).taylor(nu,0,order_expansion).trig_reduce()])
print ('E (Taylor)=',E,'\n')
# We create vin in three steps: 
# 1. A light ray emitted along the x-axis from a vertically incident muon (phi=0, see Eq. (2))
vin = vector([sin(theta_c),0,-cos(theta_c)])
# 2. Now, we rotate that vector around the y-axis toward the incidence angle mu of the muon, see Eq. (3)
RotNu = matrix(SR,3, 3, [cos(nu), 0, -sin(nu), 0, 1, 0, sin(nu), 0, cos(nu)])
vin = RotNu*vin 
print ('vin=',vin)
print ('test norm vin=',vin.norm().expand_trig().trig_reduce().simplify())
# 3. Now, we rotate that vector around the z-axis for the phi-compenent of the incidence angle mu of the muon, see Eq. (3)
RotNu = matrix(SR,3, 3, [cos(psi), -sin(psi), 0, sin(psi), cos(psi), 0, 0, 0, 1 ])
vin = RotNu*vin 
print ('vin=',vin)
print ('test norm vin=',vin.norm().expand_trig().trig_reduce().simplify())
# 4. Finally, use Rodriguez rotation formula to rotate vin around mu over an angle phi, see Eq. (4)
# https://en.wikipedia.org/wiki/Rodrigues%27_rotation_formula 
# https://math.stackexchange.com/questions/1019910/rotation-matrix-in-spherical-coordinates
# 
vin = vin*cos(phi-psi) - M.cross_product(vin)*sin(phi-psi) + M * M.dot_product(vin) * ( 1 - cos(phi-psi))
vin = vin.trig_reduce().simplify_full()

print ('\nvin=',vin,'\n LATEX:')
print ('\nvin=',latex(vin),'\n')
print ('test norm vin=',vin.norm().trig_reduce().simplify_full(),'\n')

# Taylor expansions:   
# 1 Expand nu: 
print ('vin (Taylor)',vin[0].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),'\n',
      vin[1].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),'\n',
      vin[2].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),'\n')
vin = vector([vin[0].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),
      vin[1].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),
      vin[2].collect(nu).taylor(nu,0,order_expansion).trig_reduce().simplify_full().trig_reduce()])

# 2 Expand theta_c: 
print ('vin (Taylor2)',vin[0].collect(theta_c).taylor(theta_c,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),'\n',
      vin[1].collect(theta_c).taylor(theta_c,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),'\n',
      vin[2].collect(theta_c).taylor(theta_c,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),'\n')
vin = vector([vin[0].collect(theta_c).taylor(theta_c,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),
      vin[1].collect(theta_c).taylor(theta_c,0,order_expansion).trig_reduce().simplify_full().trig_reduce(),
      vin[2].collect(theta_c).taylor(theta_c,0,order_expansion).trig_reduce().simplify_full().trig_reduce()])

# 3 Remove the mixed higher orders nu*theta_c^2 and nu^2*theta_c  by hand
vin = vector([vin[0].subs(-1/4*nu^2*theta_c*cos(-phi + 2*psi) == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/4*nu^2*theta_c*cos(-phi + 2*psi)}).trig_reduce().simplify(), 
              vin[1].subs(-1/4*nu^2*theta_c*sin(-phi + 2*psi) == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/4*nu^2*theta_c*sin(-phi + 2*psi)}).trig_reduce().simplify(),
              vin[2].subs(-1/4*nu^2*theta_c^2 == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/4*nu^2*theta_c^2}).trig_reduce().simplify()])
vin = vector([vin[0].subs(-1/4*nu^2*theta_c*cos(phi) == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/4*nu^2*theta_c*cos(phi)}).trig_reduce().simplify(), 
              vin[1].subs(-1/4*nu^2*theta_c*sin(phi) == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/4*nu^2*theta_c*sin(phi)}).trig_reduce().simplify(),
              vin[2]])
vin = vector([vin[0].subs(-1/2*nu*theta_c^2*cos(psi) == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/2*nu*theta_c^2*cos(psi)}).trig_reduce().simplify(), 
              vin[1].subs(-1/2*nu*theta_c^2*sin(psi) == eps).collect(eps).taylor(eps,0,0).subs({eps:-1/2*nu*theta_c^2*sin(psi)}).trig_reduce().simplify(),
              vin[2]])
print ('vin (Taylor3)',vin[0],'\n',vin[1],'\n',vin[2],'\n')

#vin = vector([vin[0],
#              vin[1],
#              vin[2].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,1).subs((nu*theta_c*cos(-phi + psi))==0)])

print ('vin (Taylor2)',vin[0],'\n',vin[1],'\n',vin[2])


E= (-L*cos(psi)*sin(nu) + rho*cos(pi + phi0), -L*sin(nu)*sin(psi) + rho*sin(pi + phi0), c*rho^2 + L*cos(nu))
E (Taylor)= (-L*nu*cos(psi) - rho*cos(phi0), -L*nu*sin(psi) - rho*sin(phi0), -1/2*L*nu^2 + c*rho^2 + L) 

vin= (cos(theta_c)*sin(nu) + cos(nu)*sin(theta_c), 0, -cos(nu)*cos(theta_c) + sin(nu)*sin(theta_c))
test norm vin= 1
vin= ((cos(theta_c)*sin(nu) + cos(nu)*sin(theta_c))*cos(psi), (cos(theta_c)*sin(nu) + cos(nu)*sin(theta_c))*sin(psi), -cos(nu)*cos(theta_c) + sin(nu)*sin(theta_c))
test norm vin= 1

vin= (cos(psi)*cos(theta_c)*sin(nu) + 1/2*((cos(nu) - 1)*cos(phi)*cos(2*psi) + (cos(nu) - 1)*sin(phi)*sin(2*psi) + (cos(nu) + 1)*cos(phi))*sin(theta_c), cos(theta_c)*sin(nu)*sin(psi) - 1/2*((cos(nu) - 1)*cos(2*psi)*sin(phi) - (cos(nu) - 1)*cos(phi)*sin(2*psi) - (cos(nu) + 1)*sin(phi))*sin(theta_c), -cos(nu)*cos(theta_c) + (cos(phi)*cos(psi)*sin(nu) + sin(nu)*sin(phi)*sin(psi))*sin(theta_c)) 
 LATEX:

vin= \left(\cos\left(\psi\right) \cos\left(\theta_{c}\right) \sin\left(\nu\right) 

In [3]:
D,r2,rx,ry,R1,Lmax,mm=var('D,r2,rx,ry,R1,Lmax,mm', domain='real')  
# Now search for the impact point of the photon onto the primary mirror, see Eq. (11)
# This part may take a while, but comes out correctly in the end

ip = vector([x,y,c*(x^2+y^2)])

SS=solve((E+m*vin-ip).list(),[x,y,m],solution_dict=True)[0]   # the second solution is negative 

print('vin: ',vin)
print('x1: ',SS[x],'\n')
print('y1: ',SS[y],'\n\n')

# because theta_c multiplies with l, the combination of which may become large, we expand until order 3
rx = SS[x].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,1)
ry = SS[y].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,1)

#rx = SS[x].collect(theta_c).taylor(c,0,order_expansion).trig_reduce().simplify()#.simplify()
#ry = SS[y].collect(theta_c).taylor(c,0,order_expansion).trig_reduce().simplify()#.simplify()

#rx = SS[x].collect(c).taylor(c,0,1)#.simplify()
#ry = SS[y].collect(c).taylor(c,0,1)#.simplify()

print('x2: ',rx,'\n')
print('y2: ',ry,'\n\n')

rx = rx.subs((L*nu^2 + 2*L)==2*L).trig_reduce().simplify() # remove order nu^2
ry = ry.subs((L*nu^2 + 2*L)==2*L).trig_reduce().simplify() # remove order nu^2

print('x3: ',rx,'\n')
print('y3: ',ry,'\n\n')

# check for the flat mirror case 
print('x3 (test for c=0): ',rx.subs((c==0)),'\n')
print('y3 (test for c=0): ',ry.subs((c==0)),'\n\n')

# Remove the mixed higher orders by hand (remember that L is large and practically takes away one order): 
#  L * c * nu^2*theta_c^2  --> order 4
#  L * c * nu  *theta_c^3  --> order 4
#  c*nu^2*theta_c          --> order 4

rx = rx.simplify()
ry = ry.simplify()

rx = rx.subs((5/2*L*c*nu*rho*theta_c^3*cos(phi - phi0 + psi))==0)   # minor simplifications by hand
ry = ry.subs((5/2*L*c*nu*rho*theta_c^3*sin(phi - phi0 + psi))==0)   # minor simplifications by hand

rx = rx.subs((2*L*c*nu*rho*theta_c^3*cos(-phi + phi0 + psi))==0)   # minor simplifications by hand
ry = ry.subs((2*L*c*nu*rho*theta_c^3*sin(-phi + phi0 + psi))==0)   # minor simplifications by hand

rx = rx.subs((3/2*L*c*nu*rho*theta_c^3*cos(-phi - phi0 + psi))==0)   # minor simplifications by hand
ry = ry.subs((-3/2*L*c*nu*rho*theta_c^3*sin(-phi - phi0 + psi))==0)   # minor simplifications by hand

rx = rx.subs((L*c*nu*rho*theta_c^3*cos(-3*phi + phi0 + psi))==0) # minor simplifications by hand
ry = ry.subs((-L*c*nu*rho*theta_c^3*sin(-3*phi + phi0 + psi))==0) # minor simplifications by hand

print('x5: ',rx,'\n')
print('y5: ',ry,'\n\n')

rx = rx.trig_reduce().simplify()
ry = ry.trig_reduce().simplify()

# substitutions that sagemath does not handle properly, see
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+p%29%2Bcos%28p%29
rx = rx.subs((L*c*rho*theta_c^2*cos(-2*phi+phi0))==(2*L*c*rho*theta_c^2*cos(phi)*cos(phi-phi0)-L*c*rho*theta_c^2*cos(phi0)))
# https://www.wolframalpha.com/input?i=-sin%28-2*phi+%2B+p%29%2Bsin%28p%29
ry = ry.subs((L*c*rho*theta_c^2*sin(phi0))==(2*L*c*rho*theta_c^2*sin(phi)*cos(phi-phi0)+L*c*rho*theta_c^2*sin(-2*phi+phi0)))
# https://www.wolframalpha.com/input?i=cos%28phi+-+p+%2B+psi%29+%2B+cos%28-phi+%2B+p+%2B+psi%29
rx = rx.subs((L*c*nu*rho*theta_c*cos(phi-phi0+psi))==(2*L*c*nu*rho*theta_c*cos(psi)*cos(phi-phi0)-L*c*nu*rho*theta_c*cos(-phi+phi0+psi)))
# https://www.wolframalpha.com/input?i=sin%28phi+-+p+%2B+psi%29+%2B+sin%28-phi+%2B+p+%2B+psi%29
ry = ry.subs((L*c*nu*rho*theta_c*sin(phi-phi0+psi))==(2*L*c*nu*rho*theta_c*sin(psi)*cos(phi-phi0)-L*c*nu*rho*theta_c*sin(-phi+phi0+psi)))

print('x6: ',rx,'\n')      # yields Eq. (12)
print('y6: ',ry,'\n\n')    # yields Eq. (13)

print('x5 (test for c=0): ',rx.subs(c==0),'\n')
print('y5 (test for c=0): ',ry.subs(c==0),'\n\n')




vin:  (theta_c*cos(phi) + nu*cos(psi), theta_c*sin(phi) + nu*sin(psi), nu*theta_c*cos(-phi + psi) + 1/2*nu^2 + 1/2*theta_c^2 - 1)
x1:  1/4*(theta_c^3*cos(phi) + (4*L*c*nu*cos(phi)*sin(phi)*sin(psi) + 2*nu*cos(phi)*cos(-phi + psi) - 4*(c*cos(phi0)*sin(phi)^2 - c*cos(phi)*sin(phi)*sin(phi0))*rho - (4*L*c*nu*sin(phi)^2 - nu)*cos(psi))*theta_c^2 + 4*(c*nu^2*cos(psi)*sin(phi0)*sin(psi) - c*nu^2*cos(phi0)*sin(psi)^2)*rho - (4*L*c*nu^2*cos(psi)*sin(phi)*sin(psi) - 4*L*c*nu^2*cos(phi)*sin(psi)^2 - 2*nu^2*cos(-phi + psi)*cos(psi) - 4*(c*nu*cos(psi)*sin(phi)*sin(phi0) - (2*c*nu*cos(phi0)*sin(phi) - c*nu*cos(phi)*sin(phi0))*sin(psi))*rho - (nu^2 - 2)*cos(phi))*theta_c + (nu^3 - 2*nu)*cos(psi) + sqrt(nu^4 + 4*(2*L*c*nu*cos(phi)*cos(psi) + 2*L*c*nu*sin(phi)*sin(psi) + 2*(c*cos(phi)*cos(phi0) + c*sin(phi)*sin(phi0))*rho + nu*cos(-phi + psi))*theta_c^3 + theta_c^4 + 16*(2*c^2*nu^2*cos(phi0)*cos(psi)*sin(phi0)*sin(psi) - (c^2*nu^2*sin(phi0)^2 - c^2*nu^2)*cos(psi)^2 - (c^2*nu^2*cos(phi0)^2 - c^2*nu^2)*

In [4]:
print ('Solution in latex: \n')
print('x: ',latex(rx))
print('y: ',latex(ry))
print('m: ',latex(mm))


Solution in latex: 

x:  -L^{2} c \theta_{c}^{3} \cos\left(\phi\right) + 2 \, L c \rho \theta_{c}^{2} \cos\left(\phi - \phi_{0}\right) \cos\left(\phi\right) - L^{2} c \nu \theta_{c}^{2} \cos\left(\psi\right) + 2 \, L c \nu \rho \theta_{c} \cos\left(\phi - \phi_{0}\right) \cos\left(\psi\right) + \frac{1}{2} \, L \theta_{c}^{3} \cos\left(\phi\right) + \frac{1}{2} \, L \nu \theta_{c}^{2} \cos\left(-2 \, \phi + \psi\right) + L \nu \theta_{c}^{2} \cos\left(\psi\right) + L \theta_{c} \cos\left(\phi\right) - \rho \cos\left(\phi_{0}\right)
y:  -L^{2} c \theta_{c}^{3} \sin\left(\phi\right) + 2 \, L c \rho \theta_{c}^{2} \cos\left(\phi - \phi_{0}\right) \sin\left(\phi\right) - L^{2} c \nu \theta_{c}^{2} \sin\left(\psi\right) + 2 \, L c \nu \rho \theta_{c} \cos\left(\phi - \phi_{0}\right) \sin\left(\psi\right) + \frac{1}{2} \, L \theta_{c}^{3} \sin\left(\phi\right) - \frac{1}{2} \, L \nu \theta_{c}^{2} \sin\left(-2 \, \phi + \psi\right) + L \nu \theta_{c}^{2} \sin\left(\psi\right) + L \theta_{c} 

In [5]:
# calculate the distance squared 
r2save=var('r2save', domain='real')  

r2 = (rx^2+ry^2).factor().trig_reduce().simplify().factor().simplify() # .trig_reduce().simplify()
print('vin: ',vin)
print ('r2=',r2,'\n\n')

r2 = r2.collect(c).taylor(c,0,1)

print ('r2=',r2.factor(),'\n\n')

# Remove the mixed higher orders by hand (remember that L is large and practically takes away one order): 
#  L^2 * c^2 * nu^2 * theta_c^2  --> order 4
#  L^2 * c   * nu^3 * theta_c^2  --> order 4     
#  L^2 *       nu^4 * theta_c^2  --> order 4   
#  L^2 * c^2 * nu   * theta_c^3  --> order 4 

r2 = r2.subs((2*L^2*c^2*nu^2*rho^2*theta_c^2*cos(-2*phi + 2*phi0))==0)
r2 = r2.subs((2*L^2*c^2*nu^2*rho^2*theta_c^2)==0)
r2 = r2.subs((2*L^2*c*nu^3*rho*theta_c^2*cos(-2*phi + phi0 + psi))==0)
r2 = r2.subs((2*L^2*c*nu^3*rho*theta_c^2*cos(-phi0 + psi))==0)
r2 = r2.subs((1/2*L^2*nu^4*theta_c^2*cos(-2*phi + 2*psi))==0)
r2 = r2.subs((1/2*L^2*nu^4*theta_c^2)==0)
r2 = r2.subs((1/2*L^2*nu^4*theta_c^2)==0)

r2 = r2.collect(nu).taylor(nu,0,2).factor()   
r2 = r2.collect(c).taylor(c,0,1).factor()
#r2 = r2.subs((-2*L^3*c*nu*theta_c^3*cos(-phi + psi))==0)  # ---> these corrections are important and of order two, do NOT uncomment them! 
#r2 = r2.subs((-2*L^3*c*theta_c^4)==0)    # ---> these corrections are important and of order two, do NOT uncomment them! 

# see https://www.wolframalpha.com/input?i=cos%28phi+-+2*p+%2B+a%29+%2B+cos%28-phi+%2B+a%29
r2 = r2.subs((-2*L*c*nu*rho^2*theta_c*cos(phi-2*phi0+psi))==(2*L*c*nu*rho^2*theta_c*cos(-phi+psi)-4*L*c*nu*rho^2*theta_c*cos(phi0-psi)*cos(phi-phi0)))
# see https://www.wolframalpha.com/input?i=1%2Bcos%28-2*phi%2B2*p%29
r2 = r2.subs((-2*L*c*rho^2*theta_c^2*cos(-2*phi+2*phi0))==(2*L*c*rho^2*theta_c^2-4*L*c*rho^2*theta_c^2*cos(phi-phi0)^2))

r2 = r2.subs((-4*L^3*c*nu*theta_c^5*cos(-phi + psi))==0)
r2 = r2.subs((-L^3*c*nu^2*theta_c^4*cos(-2*phi + 2*psi))==0)
r2 = r2.subs((-2*L^3*c*nu^2*theta_c^4)==0)
r2 = r2.subs((-L^3*c*theta_c^6)==0)
r2 = r2.subs((4*L^2*c*nu^2*rho*theta_c^3*cos(-phi + phi0))==0)
r2 = r2.subs((2*L^2*c*rho*theta_c^5*cos(-phi + phi0))==0)
r2 = r2.subs((L^2*c*nu^2*rho*theta_c^3*cos(-phi - phi0 + 2*psi))==0)
r2 = r2.subs((4*L^2*c*nu*rho*theta_c^4*cos(-2*phi + phi0 + psi))==0)
r2 = r2.subs((L^2*c*nu^2*rho*theta_c^3*cos(-3*phi + phi0 + 2*psi))==0)
r2 = r2.subs((4*L^2*c*nu*rho*theta_c^4*cos(-phi0 + psi))==0)
r2 = r2.subs((3/2*L^2*nu*theta_c^5*cos(-phi + psi))==0)
r2 = r2.subs((L^2*nu^2*theta_c^4*cos(-2*phi + 2*psi))==0)
r2 = r2.subs((5/4*L^2*nu^2*theta_c^4)==0)
r2 = r2.subs((1/4*L^2*theta_c^6)==0)

print ('r2 (only first and second order)=',r2,'\n\n')
print ('r2 (latex)', latex(r2))

# go directly to the maximum integration path L for the simplified solution with only first-order corrections
# R1 is radius of the primary mirror
print ('r2 before first order corrections\n\n',r2,'\n\n')

# without c , all of order nu^2 or theta^2 or nu*theta, that is, O(10^-3 or less)
r2 = r2.subs((L^2*nu^2*theta_c^2*cos(-2*phi + 2*psi))==0)
r2 = r2.subs((L^2*nu^2*theta_c^2)==0)
r2 = r2.subs((-L*nu^2*rho*theta_c*cos(-phi + phi0))==0)
r2 = r2.subs((-L*nu^2*rho*theta_c*cos(-phi - phi0 + 2*psi))==0)
r2 = r2.subs((-L*nu*rho*theta_c^2*cos(-2*phi + phi0 + psi))==0)
r2 = r2.subs((-2*L*nu*rho*theta_c^2*cos(-phi0 + psi))==0)
r2 = r2.subs((3*L^2*nu*theta_c^3*cos(-phi + psi))==0)
r2 = r2.subs((L^2*theta_c^4)==0)
r2 = r2.subs((-L*rho*theta_c^3*cos(-phi + phi0))==0)

r2save = r2

# yields Eqs. 25 - 27
print ('r2save only up to order 10^-3 corrections\n\n',r2save,'\n\n')

# with c, these are of order c*nu or c*theta, that is, O(10^-2 or less)
r2 = r2.subs((-2*L^3*c*nu*theta_c^3*cos(-phi + psi))==0)
r2 = r2.subs((-2*L^3*c*theta_c^4)==0)
r2 = r2.subs((-4*L*c*rho^2*theta_c^2*cos(phi - phi0)^2)==0)
r2 = r2.subs((6*L^2*c*rho*theta_c^3*cos(-phi + phi0))==0)
r2 = r2.subs((2*L^2*c*nu*rho*theta_c^2*cos(-2*phi + phi0 + psi))==0)
r2 = r2.subs((-4*L*c*nu*rho^2*theta_c*cos(phi - phi0)*cos(phi0 - psi))==0)
r2 = r2.subs((4*L^2*c*nu*rho*theta_c^2*cos(-phi0 + psi))==0)

print ('r2 only order 10^-2 corrections\n\n',r2,'\n\n')

print('vin: ',vin)


vin:  (theta_c*cos(phi) + nu*cos(psi), theta_c*sin(phi) + nu*sin(psi), nu*theta_c*cos(-phi + psi) + 1/2*nu^2 + 1/2*theta_c^2 - 1)
r2= 2*L^4*c^2*nu*theta_c^5*cos(-phi + psi) + L^4*c^2*nu^2*theta_c^4 + L^4*c^2*theta_c^6 - 4*L^3*c^2*nu^2*rho*theta_c^3*cos(-phi + phi0) - 4*L^3*c^2*rho*theta_c^5*cos(-phi + phi0) - 4*L^3*c^2*nu*rho*theta_c^4*cos(-2*phi + phi0 + psi) - 4*L^3*c^2*nu*rho*theta_c^4*cos(-phi0 + psi) + 2*L^2*c^2*nu*rho^2*theta_c^3*cos(phi - 2*phi0 + psi) + 4*L^2*c^2*nu*rho^2*theta_c^3*cos(-phi + psi) - 4*L^3*c*nu*theta_c^5*cos(-phi + psi) + 2*L^2*c^2*nu^2*rho^2*theta_c^2*cos(-2*phi + 2*phi0) + 2*L^2*c^2*rho^2*theta_c^4*cos(-2*phi + 2*phi0) - L^3*c*nu^2*theta_c^4*cos(-2*phi + 2*psi) + 2*L^2*c^2*nu*rho^2*theta_c^3*cos(-3*phi + 2*phi0 + psi) + 2*L^2*c^2*nu^2*rho^2*theta_c^2 - 2*L^3*c*nu^2*theta_c^4 + 2*L^2*c^2*rho^2*theta_c^4 - L^3*c*theta_c^6 + 4*L^2*c*nu^2*rho*theta_c^3*cos(-phi + phi0) + 2*L^2*c*rho*theta_c^5*cos(-phi + phi0) + L^2*c*nu^2*rho*theta_c^3*cos(-phi - phi0 + 2*psi) + 4

In [6]:
# 
# FROM HERE ON, SAGEMATH IS OVERWHELMED AND NOT ABLE TO FULL HANDLE CALCULATIONS 
# AND EXPANSIONS THAT LEAD TO THE FULL EQ. 30. 
# 
# WE SEPARATE THE NEXT-ORDER CONTRIBUTIONS INTO TWO PARTS: 
# 1)  nu=0 
# 2)  c*theta = 0
# AND SUM THEM BY HAND LATER TO GET EQ. 30
r2nonu,r2noctheta,Lmaxnonu,Lmaxnoctheta=var('r2nonu,r2noctheta,Lmaxnonu,Lmaxnoctheta', domain='real')

# first part, nu=0
r2nonu = r2save.subs(nu==0)
print ('r2 (nu=0): ',r2nonu,'\n')

TT=solve((r2nonu-R1**2),[L],solution_dict=True)[0]   # the second solution is negative. 

Lmaxnonu = TT[L]
print ('Integration limit Lmax (original, nu=0)=',Lmaxnonu,'\n\n')

print ('Integration limit Lmax (nu=0)=',Lmaxnonu.simplify(),'\n\n')
Lmaxnonu = Lmaxnonu.collect(c).taylor(c,0,1) # collect(theta_c).taylor(theta_c,order_expansion)
Lmaxnonu = Lmaxnonu.subs((cos(-phi + phi0)^2)==(1-sin(-phi0+phi)^2)) 
# leads to the 0th order and first-order corrections of Eq. 30 containing nu
print ('Integration limit Lmax (nu=0)=',Lmaxnonu,'\n\n')

# second part, c*theta=0
r2noctheta = r2save.subs((-2*L^3*c*theta_c^4)==0)
r2noctheta = r2noctheta.subs((-4*L*c*rho^2*theta_c^2*cos(phi - phi0)^2)==0)
r2noctheta = r2noctheta.subs((6*L^2*c*rho*theta_c^3*cos(-phi + phi0))==0)

print ('r2 (c*theta=0)\n\n',r2noctheta,'\n\n')

TT2=solve((r2noctheta-R1**2),[L],solution_dict=True)[0]   # the second solution is negative.

Lmaxnoctheta = TT2[L]   # .subs((cos(-phi + phi0)^2)==(1-sin(phi0-phi)^2))   # seems that Sage has no way to find out herself
print ('Integration limit Lmax (c*theta=0)=',Lmaxnoctheta,'\n\n')
Lmaxnoctheta = Lmaxnoctheta.collect(c).taylor(c,0,1)

print ('Integration limit Lmax (c*theta=0)=',Lmaxnoctheta,'\n\n')
#Lmaxnoctheta = Lmaxnoctheta.subs((-5*nu*cos(-phi + phi0)^2+nu)==(nu*sin(-phi + phi0)^2-4*nu*cos(-phi + phi0)^2))
Lmaxnoctheta = Lmaxnoctheta.subs((cos(-phi + phi0)^2-1)==(-sin(-phi0+phi)^2)) 
# https://www.wolframalpha.com/input?i=%28%284*nu*cos%28-phi+%2B+p%29%5E4+-+5*nu*cos%28-phi+%2B+p%29%5E2+%2B+nu%29*cos%28-phi+%2B+psi%29+-+2*%28nu*cos%28-phi+%2B+p%29%5E3+-+nu*cos%28-phi+%2B+p%29%29*cos%28-2*phi+%2B+p+%2B+psi%29+-+2*%28nu*cos%28-phi+%2B+p%29%5E3+-+nu*cos%28-phi+%2B+p%29%29*cos%28-p%2B+psi%29%29
Lmaxnoctheta = Lmaxnoctheta.subs(((4*nu*cos(-phi + phi0)^4 - 5*nu*cos(-phi + phi0)^2 + nu)*cos(-phi + psi) - 2*(nu*cos(-phi + phi0)^3 - nu*cos(-phi + phi0))*cos(-2*phi + phi0 + psi) - 2*(nu*cos(-phi + phi0)^3 - nu*cos(-phi + phi0))*cos(-phi0 + psi))==(nu*sin(phi-phi0)^2*cos(phi-psi)))
# https://www.wolframalpha.com/input?i=%284*S*nu*cos%28-phi%2Bp%29%5E3-3*S*nu*cos%28-phi%2Bp%29%29*cos%28-phi%2Bpsi%29-%282*S*nu*cos%28-phi%2Bp%29%5E2-S*nu%29*cos%28-2*phi+%2Bp%2Bpsi%29-2*%28S*nu*cos%28-phi%2Bp%29%5E2-S*nu%29*cos%28-p%2Bpsi%29
Lmaxnoctheta = Lmaxnoctheta.subs(((4*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0)^3-3*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0))*cos(-phi+psi)-(2*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0)^2-sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu)*cos(-2*phi+phi0+psi)-2*(sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0)^2-sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu)*cos(-phi0 + psi))==(nu*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*sin(phi-phi0)*sin(phi-psi)))
# https://www.wolframalpha.com/input?i=2*R%5E2*nu*cos%28-phi%2Bp%29*cos%28-2*phi%2Bp%2Bpsi%29+%2B+2*R%5E2*nu*cos%28-phi%2Bp%29*cos%28-p%2Bpsi%29+-+%285*R%5E2*nu*cos%28-phi%2Bp%29%5E2+-+2*R%5E2*nu%29*cos%28-phi%2Bpsi%29
Lmaxnoctheta = Lmaxnoctheta.subs((2*R1^2*nu*cos(-phi+phi0)*cos(-2*phi+phi0+psi) + 2*R1^2*nu*cos(-phi+phi0)*cos(-phi0+psi) - (5*R1^2*nu*cos(-phi+phi0)^2 - 2*R1^2*nu)*cos(-phi+psi))==(-1/2*nu*R1^2*(cos(2*phi-2*phi0)-3)*cos(phi-psi)))
# https://www.wolframalpha.com/input?i=3*S*R%5E2*nu*cos%28-phi+%2B+phi0%29*cos%28-phi+%2B+psi%29+-+S*R%5E2*nu*cos%28-2*phi+%2B+phi0+%2B+psi%29-2*S*R%5E2*nu*cos%28-phi0+%2B+psi%29
Lmaxnoctheta = Lmaxnoctheta.subs((3*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*R1^2*nu*cos(-phi+phi0)*cos(-phi+psi) - sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*R1^2*nu*cos(-2*phi+phi0+psi) - 2*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*R1^2*nu*cos(-phi0+psi))==(-nu*R1^2*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*sin(phi-phi0)*sin(phi-psi)))
# https://www.wolframalpha.com/input?i=2*nu*rho%5E4*cos%28phi-psi%29*sin%28phi-p%29%5E2%2BR%5E2*nu*rho%5E2*%28cos%282*phi-2*p%29-3%29*cos%28phi-psi%29%2B2*S*nu*rho%5E3*sin%28phi-p%29*sin%28phi-psi%29%2B2*R%5E4*nu*cos%28-phi%2Bpsi%29-2*S*nu*rho*sin%28phi-psi%29*sin%28phi%29
#Lmaxnoctheta = Lmaxnoctheta.subs((2*nu*rho^4*cos(-phi+psi)*sin(-phi+phi0)^2 + R1^2*nu*rho^2*(cos(-2*phi+2*phi0)-3)*cos(-phi+psi) - 2*sqrt(-rho^2*sin(-phi+phi0)^2+R1^2)*R1^2*nu*rho*sin(-phi+phi0)*sin(-phi+psi) + 2*sqrt(-rho^2*sin(-phi+phi0)^2+R1^2)*nu*rho^3*sin(-phi+phi0)*sin(-phi+psi) + 2*R1^4*nu*cos(-phi+psi))==(nu*(cos(phi-psi)*(rho^2*R1^2*(cos(2*phi-2*phi0)-3)+2*(rho^4*sin(phi-phi0)^2+R1^4))+2*rho*sqrt(-rho^2*sin(phi-phi0)^2 + R1^2)*sin(phi-psi)*sin(phi-phi0)*(rho^2-R1^2))))
Lmaxnoctheta = Lmaxnoctheta.subs((-2*sqrt(-rho^2*sin(-phi+phi0)^2+R1^2)*R1^2*nu*rho*sin(-phi+phi0)*sin(-phi+psi)+2*sqrt(-rho^2*sin(-phi+phi0)^2+R1^2)*nu*rho^3*sin(-phi+phi0)*sin(-phi+psi))==(2*nu*rho*sin(phi-phi0)*sin(phi-psi)*sqrt(-rho^2*sin(phi-phi0)^2 + R1^2)*(rho^2-R1^2)))
# https://www.wolframalpha.com/input?i=%28rho%5E2*cos%282*phi+-+2*p%29+%2B+2*R%5E2+-+rho%5E2%29%2F%28rho%5E2*sin%28phi+-+p%29%5E2+-+R%5E2%29
#Lmaxnoctheta = Lmaxnoctheta.subs(((rho^2*cos(2*phi-2*phi0)+2*R1^2-rho^2))==(-2*((rho^2*sin(phi - phi0)^2 - R1^2))))
Lmaxnoctheta = Lmaxnoctheta.simplify()
print ('Integration limit Lmax (c*theta=0)=',Lmaxnoctheta.simplify(),'\n\n')

Lmaxnoctheta = Lmaxnoctheta.subs((2*nu*rho^4*cos(-phi + psi)*sin(-phi + phi0)^2 + R1^2*nu*rho^2*(cos(-2*phi + 2*phi0) - 3)*cos(-phi + psi) - 2*sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*R1^2*nu*rho*sin(-phi + phi0)*sin(-phi + psi) + 2*sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*nu*rho^3*sin(-phi + phi0)*sin(-phi + psi) + 2*R1^4*nu*cos(-phi + psi))==(-(sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*rho*sin(phi - phi0)*sin(phi - psi) - (rho^2*cos(2*phi - 2*phi0) + 2*R1^2 - rho^2)*cos(phi - psi))*(R1 + rho)*(R1 - rho)*nu))
# https://www.wolframalpha.com/input?i=%28rho%5E2*cos%28-2*phi+%2B+2*p%29+%2B+2*R%5E2+-+rho%5E2%29%2F%28rho%5E2*sin%28-phi+%2B+p%29%5E2+-+R%5E2%29
Lmaxnoctheta = Lmaxnoctheta.simplify()
Lmaxnoctheta = Lmaxnoctheta.subs((rho^2*cos(-2*phi + 2*phi0) + 2*R1^2 - rho^2)==(-2*(rho^2*sin(-phi + phi0)^2 - R1^2)))
# leads to the 0th order and first-order corrections of Eq. 30 containing c
print ('Integration limit Lmax (c*theta=0)=',Lmaxnoctheta.simplify(),'\n\n')
# for the final interpretation, see https://www.wolframalpha.com/input?i=%282*rho%5E4*sin%28-phi%2Bp%29%5E2%2BR%5E2*rho%5E2*%28cos%28-2*phi%2B2*p%29-3%29%2B2*R%5E4%29%2F%28rho%5E2*sin%28-phi%2Bp%29%5E2-R%5E2%29
# and 


r2 (nu=0):  -2*L^3*c*theta_c^4 - 4*L*c*rho^2*theta_c^2*cos(phi - phi0)^2 + 6*L^2*c*rho*theta_c^3*cos(-phi + phi0) + L^2*theta_c^2 - 2*L*rho*theta_c*cos(-phi + phi0) + rho^2 

Integration limit Lmax (original, nu=0)= 1/72*(-I*sqrt(3) + 1)*(12*(2*c*rho^2*theta_c*cos(-phi + phi0)^2 + rho*cos(-phi + phi0))/(c*theta_c^3) - (6*c*rho*theta_c*cos(-phi + phi0) + 1)^2/(c^2*theta_c^4))/(-1/4*(R1^2 - rho^2)/(c*theta_c^4) - 1/12*(2*c*rho^2*theta_c*cos(-phi + phi0)^2 + rho*cos(-phi + phi0))*(6*c*rho*theta_c*cos(-phi + phi0) + 1)/(c^2*theta_c^5) + 1/216*(6*c*rho*theta_c*cos(-phi + phi0) + 1)^3/(c^3*theta_c^6) + 1/12*sqrt(-16/3*c^4*rho^6*theta_c^4*cos(-phi + phi0)^6 - 1/3*(cos(-phi + phi0)^2 - 1)*rho^2 + 1/3*(27*R1^4*c^2 + (8*c^2*cos(-phi + phi0)^4 - 36*c^2*cos(-phi + phi0)^2 + 27*c^2)*rho^4 + 18*(2*R1^2*c^2*cos(-phi + phi0)^2 - 3*R1^2*c^2)*rho^2)*theta_c^2 - 1/3*R1^2)/(c^2*theta_c^5))^(1/3) - 1/2*(I*sqrt(3) + 1)*(-1/4*(R1^2 - rho^2)/(c*theta_c^4) - 1/12*(2*c*rho^2*theta_c*cos(-phi + phi0)^2 + rho*cos

In [7]:
# Here is small sandbox for trigonometric simplifications needed to simplify "by hand" some of the above results
ttest=var('ttest',domain='real')

ttest = (4*nu*cos(-phi + phi0)^4 - 5*nu*cos(-phi + phi0)^2 + nu)*cos(-phi + psi) - 2*(nu*cos(-phi + phi0)^3 - nu*cos(-phi + phi0))*cos(-2*phi + phi0 + psi) - 2*(nu*cos(-phi + phi0)^3 - nu*cos(-phi + phi0))*cos(-phi0 + psi)
ttest=ttest.trig_expand().simplify_full().trig_reduce().simplify_full().trig_reduce().factor().simplify()
print ('\n',ttest,'\n')
#https://www.wolframalpha.com/input?i=cos%28phi+-+2*p+%2B+psi%29+-+2*cos%28-phi+%2B+psi%29+%2B+cos%28-3*phi+%2B+2*p+%2B+psi%29
ttest = ttest.subs((cos(phi - 2*phi0 + psi) - 2*cos(-phi + psi) + cos(-3*phi + 2*phi0 + psi))==(-4*sin(phi-phi0)^2*cos(phi-psi)))
print ('\n',ttest,'\n\n')

ttest = (4*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0)^3-3*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0))*cos(-phi+psi)-(2*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0)^2-sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu)*cos(-2*phi+phi0+psi)-2*(sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu*cos(-phi+phi0)^2-sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*nu)*cos(-phi0 + psi)
ttest=ttest.trig_expand().simplify_full().trig_reduce().simplify_full().trig_reduce().factor().simplify()
print ('\n',ttest,'\n')  
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+p+%2B+psi%29+-+cos%28-p+%2B+psi%29
ttest = ttest.subs((cos(-2*phi + phi0 + psi) - cos(-phi0 + psi))==(2*sin(phi0-phi)*sin(phi-psi)))
print ('\n',ttest,'\n')  
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+2*p%29+-+1
ttest = ttest.subs((cos(-2*phi + 2*phi0) - 1)==(-2*sin(phi-phi0)^2))
print ('\n',ttest,'\n\n') 

ttest = 2*R1^2*nu*cos(-phi+phi0)*cos(-2*phi+phi0+psi) + 2*R1^2*nu*cos(-phi+phi0)*cos(-phi0+psi) - (5*R1^2*nu*cos(-phi+phi0)^2 - 2*R1^2*nu)*cos(-phi+psi)
ttest=ttest.trig_expand().simplify_full().trig_reduce().simplify_full().trig_reduce().factor().simplify()
print ('\n',ttest,'\n') 
# https://www.wolframalpha.com/input?i=cos%28phi+-+2*p+%2B+psi%29+-+6*cos%28-phi+%2B+psi%29+%2B+cos%28-3*phi+%2B+2*p+%2B+psi%29
ttest = ttest.subs((cos(phi - 2*phi0 + psi) - 6*cos(-phi + psi) + cos(-3*phi + 2*phi0 + psi))==(2*(cos(2*phi-2*phi0)-3)*cos(phi-psi)))
print ('\n',ttest,'\n\n')

ttest = 3*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*R1^2*nu*cos(-phi+phi0)*cos(-phi+psi) - sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*R1^2*nu*cos(-2*phi+phi0+psi) - 2*sqrt(-rho^2*sin(phi-phi0)^2+R1^2)*R1^2*nu*cos(-phi0+psi)
ttest=ttest.trig_expand().simplify_full().trig_reduce().simplify_full().trig_reduce().factor().simplify()
print ('\n',ttest,'\n') 
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+2*p%29+-+1
ttest = ttest.subs((cos(-2*phi + 2*phi0) - 1)==(-2*sin(phi-phi0)^2))
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+p+%2B+psi%29+-+cos%28-p+%2B+psi%29
ttest = ttest.subs((cos(-2*phi + phi0 + psi) - cos(-phi0 + psi))==(2*sin(phi0-phi)*sin(phi-psi)))
print ('\n',ttest,'\n\n') 

ttest = -2*sqrt(-rho^2*sin(-phi+phi0)^2+R1^2)*R1^2*nu*rho*sin(-phi+phi0)*sin(-phi+psi)+2*sqrt(-rho^2*sin(-phi+phi0)^2+R1^2)*nu*rho^3*sin(-phi+phi0)*sin(-phi+psi)
ttest=ttest.trig_expand().simplify_full().trig_reduce().simplify_full().trig_reduce().factor().simplify()
print ('\n',ttest,'\n') 
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+2*p%29+-+1
ttest = ttest.subs((cos(-2*phi + 2*phi0) - 1)==(-2*sin(phi-phi0)^2))
# https://www.wolframalpha.com/input?i=cos%28-2*phi+%2B+p+%2B+psi%29+-+cos%28-p+%2B+psi%29
ttest = ttest.subs((cos(-2*phi + phi0 + psi) - cos(-phi0 + psi))==(2*sin(phi0-phi)*sin(phi-psi)))
print ('\n',ttest,'\n\n') 

ttest = 2*nu*rho^4*cos(-phi + psi)*sin(-phi + phi0)^2 + R1^2*nu*rho^2*(cos(-2*phi + 2*phi0) - 3)*cos(-phi + psi) - 2*sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*R1^2*nu*rho*sin(-phi + phi0)*sin(-phi + psi) + 2*sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*nu*rho^3*sin(-phi + phi0)*sin(-phi + psi) + 2*R1^4*nu*cos(-phi + psi)
ttest=ttest.trig_expand().simplify_full().trig_reduce().simplify_full().trig_reduce().factor().simplify()
print ('\n',ttest,'\n') 
# https://www.wolframalpha.com/input?i=rho%5E2*cos%28phi+-+2*p+%2B+psi%29+%2B+4*R%5E2*cos%28-phi+%2B+psi%29+-+2*rho%5E2*cos%28-phi+%2B+psi%29+%2B+rho%5E2*cos%28-3*phi+%2B+2*p+%2B+psi%29
ttest = ttest.subs((rho^2*cos(phi - 2*phi0 + psi))==(-4*R1^2*cos(-phi + psi) +2*rho^2*cos(-phi + psi) -rho^2*cos(-3*phi + 2*phi0 + psi) + 2*cos(phi-psi)*(rho^2*cos(2*phi0-2*phi)-rho^2+2*R1^2)))
# 1/2*rho^2*(cos(-2*phi + 2*p) - 1) + R^2
ttest = ttest.subs((1/2*rho^2*(cos(-2*phi + 2*phi0) - 1) + R1^2)==(R1^2-rho^2*sin(phi-phi0)^2))
ttest = ttest.subs((sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*rho*cos(-2*phi + phi0 + psi))==(sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*rho*cos(-phi0 + psi)-sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*rho*sin(phi-phi0)*sin(phi-psi)))

print ('\n',ttest,'\n\n') 


 -1/4*nu*(cos(phi - 2*phi0 + psi) - 2*cos(-phi + psi) + cos(-3*phi + 2*phi0 + psi)) 


 nu*cos(phi - psi)*sin(phi - phi0)^2 



 -1/2*sqrt(1/2*rho^2*(cos(-2*phi + 2*phi0) - 1) + R1^2)*nu*(cos(-2*phi + phi0 + psi) - cos(-phi0 + psi)) 


 -sqrt(1/2*rho^2*(cos(-2*phi + 2*phi0) - 1) + R1^2)*nu*sin(phi - psi)*sin(-phi + phi0) 


 -sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*nu*sin(phi - psi)*sin(-phi + phi0) 



 -1/4*R1^2*nu*(cos(phi - 2*phi0 + psi) - 6*cos(-phi + psi) + cos(-3*phi + 2*phi0 + psi)) 


 -1/2*R1^2*nu*(cos(2*phi - 2*phi0) - 3)*cos(phi - psi) 



 1/2*sqrt(1/2*rho^2*(cos(-2*phi + 2*phi0) - 1) + R1^2)*R1^2*nu*(cos(-2*phi + phi0 + psi) - cos(-phi0 + psi)) 


 sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*R1^2*nu*sin(phi - psi)*sin(-phi + phi0) 



 sqrt(1/2*rho^2*(cos(-2*phi + 2*phi0) - 1) + R1^2)*(R1 + rho)*(R1 - rho)*nu*rho*(cos(-2*phi + phi0 + psi) - cos(-phi0 + psi)) 


 2*sqrt(-rho^2*sin(phi - phi0)^2 + R1^2)*(R1 + rho)*(R1 - rho)*nu*rho*sin(phi - psi)*sin(-phi + phi0) 



 1/2*(rho^2*c

In [8]:
# calculate the angle alpha
talpha,calpha2=var('talpha,calpha2', domain='real')  # definitions of tan(alpha) and cos(2*alpha)

# calculate first the tan(alpha) and simplify
talpha = (ry/rx).factor().trig_reduce().simplify().factor().simplify() # .trig_reduce().simplify()
print ('talpha=',talpha,'\n\n')

# Remove the mixed higher orders by hand (remember that L is large and practically takes away one order): 
#  L^2 * c * nu * theta_c^2  --> order 2
#  L   * c * nu * theta_c    --> order 2     
#  L   * c *      theta_c^2  --> order 2   
#  L   *    nu^2* theta_c    --> order 2 
#  L   *     nu   theta_c^2  --> order 2 
#  L   *          theta_c^2  --> order 2 
# 
# unfortunatley, the order of these commands matters...
talpha = talpha.subs((2*L^2*c*theta_c^3*sin(phi))==0)
talpha = talpha.subs((2*L^2*c*nu*theta_c^2*sin(psi))==0)
talpha = talpha.subs((-2*L*c*nu*rho*theta_c*sin(phi - phi0 + psi))==0)
talpha = talpha.subs((-2*L*c*nu*rho*theta_c*sin(-phi + phi0 + psi))==0)
talpha = talpha.subs((2*L*c*nu*rho*theta_c*sin(phi - phi0 + psi))==0)
talpha = talpha.subs((2*L*c*nu*rho*theta_c*sin(-phi + phi0 + psi))==0)
talpha = talpha.subs((2*L*c*rho*theta_c^2*sin(phi0))==0)
talpha = talpha.subs((-2*L*c*rho*theta_c^2*sin(phi0))==0)
talpha = talpha.subs((2*L*c*rho*theta_c^2*sin(-2*phi + phi0))==0)
talpha = talpha.subs((L*nu^2*theta_c*sin(phi))==0)
talpha = talpha.subs((L*nu^2*theta_c*sin(-phi + 2*psi))==0)
talpha = talpha.subs((L*nu*theta_c^2*sin(-2*phi + psi))==0)
talpha = talpha.subs((2*L*nu*theta_c^2*sin(psi))==0)
talpha = talpha.subs((L*nu*theta_c^2*sin(psi))==0)
talpha = talpha.subs((L*theta_c^3*sin(phi))==0)
talpha = talpha.subs((L*nu*theta_c^2*sin(-2*phi + psi))==0)

talpha = talpha.subs((2*L^2*c*nu*theta_c^2*cos(psi))==0)
talpha = talpha.subs((2*L*c*nu*rho*theta_c*cos(-phi + phi0 + psi))==0)


talpha = talpha.subs((L*nu^2*theta_c*cos(phi))==0)
talpha = talpha.subs((L*nu^2*theta_c*cos(-phi + 2*psi))==0)


talpha = talpha.subs((2*L*nu*theta_c^2*cos(psi))==0)
talpha = talpha.subs((2*L^2*c*theta_c^3*cos(phi))==0)
talpha = talpha.subs((2*L*c*nu*rho*theta_c*cos(-phi + phi0 + psi))==0)
talpha = talpha.subs((2*L*c*nu*rho*theta_c*cos(phi - phi0 + psi))==0)
talpha = talpha.subs((2*L*c*rho*theta_c^2*cos(-2*phi + phi0))==0)
talpha = talpha.subs((2*L*c*rho*theta_c^2*cos(phi0))==0)
talpha = talpha.subs((L*theta_c^3*cos(phi))==0)
talpha = talpha.subs((L*nu*theta_c^2*cos(-2*phi + psi))==0)
talpha = talpha.subs((L*nu*theta_c^2*cos(psi))==0)
#r2 = r2.subs((1/2*L^2*nu^4*theta_c^2)==0)
#r2 = r2.subs((1/2*L^2*nu^4*theta_c^2)==0)

print ('talpha=',talpha,'\n\n')
print ('latex(talpha)=',latex(talpha),'\n\n')

print ('alpha=',atan(talpha),'\n\n')
print ('sin(alpha)=',sin(atan(talpha)).factor().simplify(),'\n\n')
print ('latex(sin(alpha))=',latex(sin(atan(talpha)).factor().simplify()),'\n\n')
print ('cos(alpha)=',cos(atan(talpha)).factor().trig_reduce().simplify(),'\n\n')
print ('latex(cos(alpha))=',latex(cos(atan(talpha)).factor().trig_reduce().simplify()),'\n\n')
calpha2 = (cos(atan(talpha)).factor().trig_reduce().simplify())**2
print ('cos^2(alpha)=',calpha2,'\n\n')
print ('latex(cos^2(alpha))=',latex(calpha2),'\n\n')

talpha= (2*L^2*c*theta_c^3*sin(phi) + 2*L^2*c*nu*theta_c^2*sin(psi) - 2*L*c*nu*rho*theta_c*sin(phi - phi0 + psi) - 2*L*c*nu*rho*theta_c*sin(-phi + phi0 + psi) + 2*L*c*rho*theta_c^2*sin(-2*phi + phi0) - 2*L*c*rho*theta_c^2*sin(phi0) - L*theta_c^3*sin(phi) + L*nu*theta_c^2*sin(-2*phi + psi) - 2*L*nu*theta_c^2*sin(psi) - 2*L*theta_c*sin(phi) + 2*rho*sin(phi0))/(2*L^2*c*theta_c^3*cos(phi) + 2*L^2*c*nu*theta_c^2*cos(psi) - 2*L*c*nu*rho*theta_c*cos(phi - phi0 + psi) - 2*L*c*nu*rho*theta_c*cos(-phi + phi0 + psi) - 2*L*c*rho*theta_c^2*cos(-2*phi + phi0) - 2*L*c*rho*theta_c^2*cos(phi0) - L*theta_c^3*cos(phi) - L*nu*theta_c^2*cos(-2*phi + psi) - 2*L*nu*theta_c^2*cos(psi) - 2*L*theta_c*cos(phi) + 2*rho*cos(phi0)) 


talpha= (L*theta_c*sin(phi) - rho*sin(phi0))/(L*theta_c*cos(phi) - rho*cos(phi0)) 


latex(talpha)= \frac{L \theta_{c} \sin\left(\phi\right) - \rho \sin\left(\phi_{0}\right)}{L \theta_{c} \cos\left(\phi\right) - \rho \cos\left(\phi_{0}\right)} 


alpha= arctan((L*theta_c*sin(phi) - rh

cos(alpha)= abs(L^2*c*theta_c^3*cos(phi) + L^2*c*nu*theta_c^2*cos(psi) - L*c*nu*rho*theta_c*cos(phi - phi0 + psi) - L*c*nu*rho*theta_c*cos(-phi + phi0 + psi) - L*c*rho*theta_c^2*cos(-2*phi + phi0) - L*c*rho*theta_c^2*cos(phi0) - L*theta_c*cos(phi) + rho*cos(phi0))/sqrt(2*L^4*c^2*nu*theta_c^5*cos(-phi + psi) + L^4*c^2*nu^2*theta_c^4 + L^4*c^2*theta_c^6 - 4*L^3*c^2*nu^2*rho*theta_c^3*cos(-phi + phi0) - 4*L^3*c^2*rho*theta_c^5*cos(-phi + phi0) - 4*L^3*c^2*nu*rho*theta_c^4*cos(-2*phi + phi0 + psi) - 4*L^3*c^2*nu*rho*theta_c^4*cos(-phi0 + psi) + 2*L^2*c^2*nu*rho^2*theta_c^3*cos(phi - 2*phi0 + psi) + 4*L^2*c^2*nu*rho^2*theta_c^3*cos(-phi + psi) + 2*L^2*c^2*nu^2*rho^2*theta_c^2*cos(-2*phi + 2*phi0) + 2*L^2*c^2*rho^2*theta_c^4*cos(-2*phi + 2*phi0) + 2*L^2*c^2*nu*rho^2*theta_c^3*cos(-3*phi + 2*phi0 + psi) + 2*L^2*c^2*nu^2*rho^2*theta_c^2 + 2*L^2*c^2*rho^2*theta_c^4 - 2*L^3*c*nu*theta_c^3*cos(-phi + psi) - 2*L^3*c*theta_c^4 + 6*L^2*c*rho*theta_c^3*cos(-phi + phi0) + 2*L^2*c*nu*rho*theta_c^2*cos(

In [9]:
# new helper variable for the chord test for the simplified solution with vertical muon and flat mirror
DD=var('DD',domain='real')
DD = r2 
DD = DD.subs({L:D/theta_c}).subs(c==0).subs(nu==0)
print ('Chord = ',DD,'\n')

# now test for the maximum value of the chord D for integration, just for comparison
TT=solve((DD-R1**2),[D],solution_dict=True)[1]   # la first solucion is the negative one. 

print ('Chord D=',TT[D].simplify().subs((cos(-phi + phi0)^2 - 1)==-sin(-phi+phi0)^2),'\n')
print ('Chord D (latex)',latex(TT[D].simplify().subs((cos(-phi + phi0)^2 - 1)==-sin(-phi+phi0)^2)))
# .. which is the formula Eq. (8) of Ahnen et al., 2019, if we replace rho by rho_R * R, so up to here everything OK 

Chord =  -2*D*rho*cos(-phi + phi0) + D^2 + rho^2 

Chord D= rho*cos(-phi + phi0) + sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2) 

Chord D (latex) \rho \cos\left(-\phi + \phi_{0}\right) + \sqrt{-\rho^{2} \sin\left(-\phi + \phi_{0}\right)^{2} + R_{1}^{2}}


In [10]:
# go directly to the maximum integration path L for the simplified solution with only first-order corrections
# R1 is radius of the primary mirror
print ('r2 before first order corrections\n\n',r2,'\n\n')
r2 = r2.subs((L^2*nu^2*theta_c^2*cos(-2*phi + 2*psi))==0)
r2 = r2.subs((L^2*nu^2*theta_c^2)==0)
r2 = r2.subs((-L*nu^2*rho*theta_c*cos(-phi + phi0))==0)
r2 = r2.subs((-L*nu^2*rho*theta_c*cos(-phi - phi0 + 2*psi))==0)
r2 = r2.subs((-L*nu*rho*theta_c^2*cos(-2*phi + phi0 + psi))==0)
r2 = r2.subs((-2*L*nu*rho*theta_c^2*cos(-phi0 + psi))==0)
print ('r2 only first order corrections\n\n',r2,'\n\n')

print('vin: ',vin)

r2 before first order corrections

 L^2*theta_c^2 - 2*L*rho*theta_c*cos(-phi + phi0) + rho^2 


r2 only first order corrections

 L^2*theta_c^2 - 2*L*rho*theta_c*cos(-phi + phi0) + rho^2 


vin:  (theta_c*cos(phi) + nu*cos(psi), theta_c*sin(phi) + nu*sin(psi), nu*theta_c*cos(-phi + psi) + 1/2*nu^2 + 1/2*theta_c^2 - 1)


In [11]:
# This step takes quite some time, please be patient, normally it does not 
# converge, that is why the two separate expansions with Lmaxnonu and Lmaxnoctheta 
# have been made earlier.
TT=solve((r2save-R1**2),[L],solution_dict=True)[0]   # the second solution is negative. 

Lmax = TT[L]


print ('Lmax=',Lmax,'\n\n')
#Lmax = TT[L].collect(nu).taylor(nu,0,order_expansion)
#print ('Lmax=',Lmax,'\n\n')

Lmax = Lmax.subs((cos(-phi + phi0)^2 - 1)==-sin(phi0-phi)^2)
Lmax = Lmax.subs((2*cos(-phi + phi0)^3 - cos(-phi + phi0)*cos(-2*phi + 2*phi0) - cos(-phi + phi0))==0) # check https://www.wolframalpha.com/input?i=2*cos%28-phi+%2B+p%29%5E3+-+cos%28-phi+%2B+p%29*cos%28-2*phi+%2B+2*p%29+-+cos%28-phi+%2B+p%29
Lmax = Lmax.subs((2*cos(-phi + phi0)^2 - cos(-2*phi + 2*phi0))==1)     # https://www.wolframalpha.com/input?i=2*cos%28-phi+%2B+p%29%5E2+-+cos%28-2*phi+%2B+2*p%29

Lmax = Lmax.subs((nu*cos(-phi + phi0)^2 - nu)==-nu*sin(phi0-phi)^2)
Lmax = Lmax.subs((nu*cos(-phi + phi0)^3 - nu*cos(-phi + phi0))==-nu*cos(-phi + phi0)*sin(phi0 - phi)^2)
Lmax = Lmax.subs((cos(-2*phi + 2*phi0)*sin(-phi + phi0)^2 - cos(-phi + phi0)^2 + 1)==sin(-phi + phi0)^2*(1+cos(-2*phi + 2*phi0)))
Lmax = Lmax.subs((nu*cos(-phi + phi0)*cos(-2*phi + phi0 + psi)*sin(-phi + phi0)^2 + nu*cos(-phi + phi0)*cos(-phi0 + psi)*sin(-phi + phi0)^2)==(1/2*nu*cos(phi-psi)*sin(2*phi-2*phi0)^2))
Lmax = Lmax.subs((R1^2*nu*cos(-phi + phi0)*cos(-2*phi + phi0 + psi) + R1^2*nu*cos(-phi + phi0)*cos(-phi0 + psi))==(2*R1^2*nu*cos(phi-phi0)^2*cos(phi-psi)))
Lmax = Lmax.subs((4*cos(-phi + phi0)^3 - 3*cos(-phi + phi0))==(cos(3*phi-3*phi0)))
Lmax = Lmax.subs((cos(-phi + phi0)^4 - cos(-phi + phi0)^2)==(-1/4*sin(2*phi-2*phi0)^2))

Lmax = Lmax.collect(c).taylor(c,0,1)
print ('Lmax=',Lmax,'\n\n')
Lmax = Lmax.collect(theta_c).taylor(theta_c,0,4)
print ('Lmax=',Lmax,'\n\n')
#print ('Lmax=',Lmax.simplify_full(),'\n\n')
Lmax = Lmax.subs((rho^2*cos(-phi + phi0)^2 + R1^2 - rho^2)==(R1^2-rho^2*sin(phi-phi0)^2))
# see https://www.wolframalpha.com/input?i=%284*cos%28-phi%2Bp%29%5E3+-+3*cos%28-phi%2Bp%29%29*cos%28-phi%2Bpsi%29+-+%282*cos%28-phi%2Bp%29%5E2-1%29*cos%28-2*phi%2Bp%2Bpsi%29-2*%28cos%28-phi%2Bp%29%5E2-1%29*cos%28-p%2Bpsi%29
Lmax = Lmax.subs(((4*c*nu*cos(-phi + phi0)^3 - 3*c*nu*cos(-phi + phi0))*cos(-phi + psi) - (2*c*nu*cos(-phi + phi0)^2 - c*nu)*cos(-2*phi + phi0 + psi) - 2*(c*nu*cos(-phi + phi0)^2 - c*nu)*cos(-phi0 + psi))==(c*nu*sin(phi-phi0)*sin(phi-psi)))
# see https://www.wolframalpha.com/input?i=3*cos%28-phi+%2B+p%29*cos%28-phi+%2B+psi%29+-+cos%28-2*phi+%2B+p+%2B+psi%29+-+2*cos%28-p+%2B+psi%29
Lmax = Lmax.subs((3*R1^2*c*nu*cos(-phi + phi0)*cos(-phi + psi) - R1^2*c*nu*cos(-2*phi + phi0 + psi) - 2*R1^2*c*nu*cos(-phi0 + psi))==(-R1^2*c*nu*sin(phi-phi0)*sin(phi-psi)))
# see https://www.wolframalpha.com/input?i=2*cos%28-phi+%2B+p%29*cos%28-2*phi+%2B+p+%2B+psi%29+%2B+2*cos%28-phi+%2B+p%29*cos%28-p+%2B+psi%29+-+%284*cos%28-phi+%2B+p%29%5E2+-+1%29*cos%28-phi+%2B+psi%29
Lmax = Lmax.subs((2*c*nu*cos(-phi + phi0)*cos(-2*phi + phi0 + psi) + 2*c*nu*cos(-phi + phi0)*cos(-phi0 + psi) - (4*c*nu*cos(-phi + phi0)^2 - c*nu)*cos(-phi + psi))==(c*nu*cos(phi-psi)))
Lmax = Lmax.subs((cos(-phi + phi0)^2 - 1)==(-sin(phi-phi0)^2))

print ('Lmax=',Lmax,'\n\n')   # provides Eq. 30 

print ('Lmax (simplify)=',Lmax.simplify(),'\n\n') # provides Eq. 30 

print ('Lmax (factor)=',Lmax.factor().simplify(),'\n\n')   # factor seems not to help much
print ('Lmax for nu=0',Lmax.subs(nu==0).simplify(),'\n\n')

print('vin: ',vin)

Lmax= 1/72*(-I*sqrt(3) + 1)*(12*(2*(rho^2*theta_c*cos(-phi + phi0)^2 + nu*rho^2*cos(-phi + phi0)*cos(-phi0 + psi))*c + rho*cos(-phi + phi0))/((nu*theta_c^2*cos(-phi + psi) + theta_c^3)*c) - (2*(3*rho*theta_c*cos(-phi + phi0) + (rho*cos(-2*phi + phi0 + psi) + 2*rho*cos(-phi0 + psi))*nu)*c + 1)^2/((nu*theta_c*cos(-phi + psi) + theta_c^2)^2*c^2))/(-1/4*(R1^2 - rho^2)/((nu*theta_c^3*cos(-phi + psi) + theta_c^4)*c) - 1/12*(2*(rho^2*theta_c*cos(-phi + phi0)^2 + nu*rho^2*cos(-phi + phi0)*cos(-phi0 + psi))*c + rho*cos(-phi + phi0))*(2*(3*rho*theta_c*cos(-phi + phi0) + (rho*cos(-2*phi + phi0 + psi) + 2*rho*cos(-phi0 + psi))*nu)*c + 1)/((nu*theta_c^2*cos(-phi + psi) + theta_c^3)*(nu*theta_c*cos(-phi + psi) + theta_c^2)*c^2) + 1/216*(2*(3*rho*theta_c*cos(-phi + phi0) + (rho*cos(-2*phi + phi0 + psi) + 2*rho*cos(-phi0 + psi))*nu)*c + 1)^3/((nu*theta_c*cos(-phi + psi) + theta_c^2)^3*c^3) + 1/12*sqrt(-16/3*c^4*rho^6*theta_c^4*cos(-phi + phi0)^6 + 32/3*(4*c^4*nu*cos(-phi + phi0)^6*cos(-phi + psi) - 3*

In [12]:
print ('Lmax (latex)',latex(Lmax),'\n\n')
print ('dN/dphi=',(Lmax*theta_c^2).factor().simplify(), ' * (alpha/hc) * (B_mu)')

Lmax (latex) R_{1}^{2} c - c \rho^{2} - \frac{R_{1}^{2} c \nu \rho \sin\left(\phi - \phi_{0}\right) \sin\left(\phi - \psi\right) - c \nu \rho^{3} \sin\left(\phi - \phi_{0}\right) \sin\left(\phi - \psi\right) + {\left(c \nu \rho^{2} \cos\left(\phi - \psi\right) - R_{1}^{2} c \nu \cos\left(-\phi + \psi\right) - \rho \cos\left(-\phi + \phi_{0}\right) - \sqrt{-\rho^{2} \sin\left(\phi - \phi_{0}\right)^{2} + R_{1}^{2}}\right)} \sqrt{-\rho^{2} \sin\left(\phi - \phi_{0}\right)^{2} + R_{1}^{2}}}{\sqrt{-\rho^{2} \sin\left(\phi - \phi_{0}\right)^{2} + R_{1}^{2}} \theta_{c}} 


dN/dphi= -(R1^2*c*nu*rho*sin(-phi + phi0)*sin(-phi + psi) - c*nu*rho^3*sin(-phi + phi0)*sin(-phi + psi) - sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*R1^2*c*nu*cos(-phi + psi) + sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*c*nu*rho^2*cos(-phi + psi) - sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*R1^2*c*theta_c + sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*c*rho^2*theta_c + rho^2*sin(-phi + phi0)^2 - sqrt(-rho^2*sin(-phi + phi0)^2 + R1^2)*rho*co

In [13]:
TT2=solve((r2save-R1**2),[L],solution_dict=True)[1]   # try the other solution

Lmax2 = TT2[L].subs((cos(-phi + phi0)^2 - 1)==-sin(phi0-phi)^2)   # seems that Sage has no way to find out herself

print ('Lmax2=',Lmax2,'\n\n')
#Lmax = TT[L].collect(nu).taylor(nu,0,order_expansion)
#print ('Lmax=',Lmax,'\n\n')

Lmax2 = Lmax2.subs((cos(-phi + phi0)^2 - 1)==-sin(phi0-phi)^2)
Lmax2 = Lmax2.subs((2*cos(-phi + phi0)^3 - cos(-phi + phi0)*cos(-2*phi + 2*phi0) - cos(-phi + phi0))==0) # check https://www.wolframalpha.com/input?i=2*cos%28-phi+%2B+p%29%5E3+-+cos%28-phi+%2B+p%29*cos%28-2*phi+%2B+2*p%29+-+cos%28-phi+%2B+p%29
Lmax2 = Lmax2.subs((2*cos(-phi + phi0)^2 - cos(-2*phi + 2*phi0))==1)     # https://www.wolframalpha.com/input?i=2*cos%28-phi+%2B+p%29%5E2+-+cos%28-2*phi+%2B+2*p%29

Lmax2 = Lmax2.subs((nu*cos(-phi + phi0)^2 - nu)==-nu*sin(phi0-phi)^2)
Lmax2 = Lmax2.subs((nu*cos(-phi + phi0)^3 - nu*cos(-phi + phi0))==-nu*cos(-phi + phi0)*sin(phi0 - phi)^2)
Lmax2 = Lmax2.subs((cos(-2*phi + 2*phi0)*sin(-phi + phi0)^2 - cos(-phi + phi0)^2 + 1)==sin(-phi + phi0)^2*(1+cos(-2*phi + 2*phi0)))
Lmax2 = Lmax2.subs((nu*cos(-phi + phi0)*cos(-2*phi + phi0 + psi)*sin(-phi + phi0)^2 + nu*cos(-phi + phi0)*cos(-phi0 + psi)*sin(-phi + phi0)^2)==(1/2*nu*cos(phi-psi)*sin(2*phi-2*phi0)^2))
Lmax2 = Lmax2.subs((R1^2*nu*cos(-phi + phi0)*cos(-2*phi + phi0 + psi) + R1^2*nu*cos(-phi + phi0)*cos(-phi0 + psi))==(2*R1^2*nu*cos(phi-phi0)^2*cos(phi-psi)))
Lmax2 = Lmax2.subs((4*cos(-phi + phi0)^3 - 3*cos(-phi + phi0))==(cos(3*phi-3*phi0)))
Lmax2 = Lmax2.subs((cos(-phi + phi0)^4 - cos(-phi + phi0)^2)==(-1/4*sin(2*phi-2*phi0)^2))

Lmax2 = Lmax2.collect(c).taylor(c,0,1)
print ('Lmax2=',Lmax2,'\n\n')
Lmax2 = Lmax2.collect(theta_c).taylor(theta_c,0,4)
print ('Lmax2=',Lmax2,'\n\n')
#print ('Lmax=',Lmax.simplify_full(),'\n\n')
Lmax2 = Lmax2.subs((rho^2*cos(-phi + phi0)^2 + R1^2 - rho^2)==(R1^2-rho^2*sin(phi-phi0)^2))
# see https://www.wolframalpha.com/input?i=%284*cos%28-phi%2Bp%29%5E3+-+3*cos%28-phi%2Bp%29%29*cos%28-phi%2Bpsi%29+-+%282*cos%28-phi%2Bp%29%5E2-1%29*cos%28-2*phi%2Bp%2Bpsi%29-2*%28cos%28-phi%2Bp%29%5E2-1%29*cos%28-p%2Bpsi%29
Lmax2 = Lmax2.subs(((4*c*nu*cos(-phi + phi0)^3 - 3*c*nu*cos(-phi + phi0))*cos(-phi + psi) - (2*c*nu*cos(-phi + phi0)^2 - c*nu)*cos(-2*phi + phi0 + psi) - 2*(c*nu*cos(-phi + phi0)^2 - c*nu)*cos(-phi0 + psi))==(c*nu*sin(phi-phi0)*sin(phi-psi)))
# see https://www.wolframalpha.com/input?i=3*cos%28-phi+%2B+p%29*cos%28-phi+%2B+psi%29+-+cos%28-2*phi+%2B+p+%2B+psi%29+-+2*cos%28-p+%2B+psi%29
Lmax2 = Lmax2.subs((3*R1^2*c*nu*cos(-phi + phi0)*cos(-phi + psi) - R1^2*c*nu*cos(-2*phi + phi0 + psi) - 2*R1^2*c*nu*cos(-phi0 + psi))==(-R1^2*c*nu*sin(phi-phi0)*sin(phi-psi)))
# see https://www.wolframalpha.com/input?i=2*cos%28-phi+%2B+p%29*cos%28-2*phi+%2B+p+%2B+psi%29+%2B+2*cos%28-phi+%2B+p%29*cos%28-p+%2B+psi%29+-+%284*cos%28-phi+%2B+p%29%5E2+-+1%29*cos%28-phi+%2B+psi%29
Lmax2 = Lmax2.subs((2*c*nu*cos(-phi + phi0)*cos(-2*phi + phi0 + psi) + 2*c*nu*cos(-phi + phi0)*cos(-phi0 + psi) - (4*c*nu*cos(-phi + phi0)^2 - c*nu)*cos(-phi + psi))==(c*nu*cos(phi-psi)))
Lmax2 = Lmax2.subs((cos(-phi + phi0)^2 - 1)==(-sin(phi-phi0)^2))

print ('Lmax2=',Lmax2,'\n\n')   # provides Eq. 30 

print ('Lmax2 for nu=0',Lmax2.subs(nu==0).simplify(),'\n\n')

Lmax2= 1/72*(I*sqrt(3) + 1)*(12*(2*(rho^2*theta_c*cos(-phi + phi0)^2 + nu*rho^2*cos(-phi + phi0)*cos(-phi0 + psi))*c + rho*cos(-phi + phi0))/((nu*theta_c^2*cos(-phi + psi) + theta_c^3)*c) - (2*(3*rho*theta_c*cos(-phi + phi0) + (rho*cos(-2*phi + phi0 + psi) + 2*rho*cos(-phi0 + psi))*nu)*c + 1)^2/((nu*theta_c*cos(-phi + psi) + theta_c^2)^2*c^2))/(-1/4*(R1^2 - rho^2)/((nu*theta_c^3*cos(-phi + psi) + theta_c^4)*c) - 1/12*(2*(rho^2*theta_c*cos(-phi + phi0)^2 + nu*rho^2*cos(-phi + phi0)*cos(-phi0 + psi))*c + rho*cos(-phi + phi0))*(2*(3*rho*theta_c*cos(-phi + phi0) + (rho*cos(-2*phi + phi0 + psi) + 2*rho*cos(-phi0 + psi))*nu)*c + 1)/((nu*theta_c^2*cos(-phi + psi) + theta_c^3)*(nu*theta_c*cos(-phi + psi) + theta_c^2)*c^2) + 1/216*(2*(3*rho*theta_c*cos(-phi + phi0) + (rho*cos(-2*phi + phi0 + psi) + 2*rho*cos(-phi0 + psi))*nu)*c + 1)^3/((nu*theta_c*cos(-phi + psi) + theta_c^2)^3*c^3) + 1/12*sqrt(-16/3*c^4*rho^6*theta_c^4*cos(-phi + phi0)^6 + 32/3*(4*c^4*nu*cos(-phi + phi0)^6*cos(-phi + psi) - 3*

In [14]:
print ('vin=',vin)

vin= (theta_c*cos(phi) + nu*cos(psi), theta_c*sin(phi) + nu*sin(psi), nu*theta_c*cos(-phi + psi) + 1/2*nu^2 + 1/2*theta_c^2 - 1)


In [15]:
# Reflect the MUON on the primary mirror, as if it were a photon (just for A SHORT ABERRATION TEST) and check how it is seen in the camera
rxm1,rym1 = var('rxm1,rym1', domain='real')  # keep here vin only to zero and first order 
rxm1 = I[0]
rym1 = I[1]
print('rxm1=', rxm1)
print('rym1=', rym1)

rxm1 = rho*cos(phi0)
rym1 = rho*sin(phi0)
II = vector([rxm1,rym1,c*rho^2])
print ('II=',II)

nm=var('nm', domain='real')  

print('rxm1=', rxm1)
print('rym1=', rym1)

MM = vector([nu*cos(psi), nu*sin(psi), -1 ])  # muon vector to first order 
print('M=',MM)

# nm is the vector normal to the paraboloide n=(-x/(2*f),-y/(2*f),1) = (-2cx,-2cy,1)
nm=vector([-2*c*rxm1,-2*c*rym1,1])
print('nm: ',nm,'\n\n')

vout_mx,vout_my,vout_mz=var('vout_mx,vout_my,vout_mz', domain='real') 

# Now reflect the muon vector MM on the mirror at impact point II  vout=vint-(2(vin·n)/|n|^2)
voutm=MM-2*MM.dot_product(nm)*nm/(nm.norm()^2)

vout_mx = voutm[0]#.collect(c).taylor(c,0,order_expansion)#.simplify()#.trig_reduce().simplify().factor().simplify()
vout_my = voutm[1]#.collect(c).taylor(c,0,order_expansion)#.simplify()#.trig_reduce().simplify().factor().simplify()
vout_mz = voutm[2]#.collect(c).taylor(c,0,order_expansion)#.simplify()#.trig_reduce().simplify().factor().simplify()

print('full voutm:',vout_mx,'\n\n',vout_my,'\n\n',vout_mz,'\n\n')

vout_mx = vout_mx.collect(c).taylor(c,0,2).trig_reduce().simplify()
vout_my = vout_my.collect(c).taylor(c,0,2).trig_reduce().simplify()
vout_mz = vout_mz.collect(c).taylor(c,0,2).trig_reduce().simplify()

print('Taylor-expanded voutm:',vout_mx,'\n\n',vout_my,'\n\n',vout_mz,'\n\n')

voutm = vector([vout_mx,vout_my,vout_mz])

sm,fmx,fmy,fmm,fmr2,xm,ym=var('sm,fmx,fmy,fmm,fmr2,xm,ym', domain='real')  
# Now we study the photon's (muon's) crossing point with the focal plane

Fplanem = vector([xm,ym,1/(4*c)]) # Pla focal c = 1/(4f)
print ('II=',II)

EqPlaFoc=solve((II+sm*voutm-Fplanem).list(),[sm,xm,ym],solution_dict=True)[0]  # only one solution possible here
 
fmm = EqPlaFoc[sm]#.trig_reduce()
fmx = EqPlaFoc[xm]#.trig_reduce().collect(theta_c).taylor(theta_c,0,1).simplify().collect(nu).taylor(nu,0,1).simplify().trig_expand().factor().simplify_full().trig_reduce().simplify()
fmy = EqPlaFoc[ym]#.trig_reduce().collect(theta_c).taylor(theta_c,0,1).simplify().collect(nu).taylor(nu,0,1).simplify().trig_expand().factor().simplify_full().trig_reduce().simplify()

print('m: ',fmm,'\n')
print('x: ',fmx,'\n')
print('y: ',fmy,'\n')

fmx = fmx.collect(nu).taylor(nu,0,1).trig_reduce().simplify()
fmy = fmy.collect(nu).taylor(nu,0,1).trig_reduce().simplify()
print('nu-Taylor-expanded x: ',fmx,'\n')
print('nu-Taylor-expanded y: ',fmy,'\n')

fmm = fmm.factor().collect(c).taylor(c,0,1).collect(nu).taylor(nu,0,1).factor().trig_reduce().simplify() 
fmx = fmx.factor().collect(c).taylor(c,0,1).trig_reduce().simplify()   # need to go to 3rd order, because the overall gets divided by c
fmy = fmy.factor().collect(c).taylor(c,0,1).trig_reduce().simplify()

print('Taylor-expanded m: ',fmm,'\n')
print('Taylor-expanded x: ',fmx,'\n')
print('Taylor-expanded y: ',fmy,'\n')

fmx = fmx.subs(psi==0).collect(c).taylor(c,0,1)
fmy = fmy.subs(psi==0).factor().collect(c).taylor(c,0,1)

print('x (psi=0):                            ',fmx,'\nCompare with theory (tangential coma): 1/(4F)*nu*rho^2 * ( cos(2phi_0) + 2)\n\n')
print('y (psi=0):                            ',fmy,'\nCompare with theory (sagittal coma)  : 1/(4F)*nu*rho^2 * sin(2phi_0)\n\n')

RotNu2 = matrix(SR,2,2,[cos(psi),-sin(psi),sin(psi),cos(psi)])

C = vector([fmx,fmy])
C = RotNu2*C

fmx = C[0].trig_reduce().simplify().factor().simplify()
fmy = C[1].trig_reduce().simplify().factor().simplify()

print('x (rotated back to psi): ',fmx,'\n')
print('y (rotated back to psi): ',fmy,'\n')


#fmr2 = ((fmx-1/(4*c)*nu*cos(psi))^2+(fmy-1/(4*c)*nu*sin(psi))^2).simplify().factor().simplify().collect(c).taylor(c,0,2).trig_reduce().simplify()
#fmr  = sqrt(fmr2).trig_reduce().simplify().collect(c).taylor(c,0,1)
#print('r2: ',fmr2,'\n')
#print('r: ',fmr,'\n')
print('vin: ',vin)

rxm1= 0
rym1= 1
II= (rho*cos(phi0), rho*sin(phi0), c*rho^2)
rxm1= rho*cos(phi0)
rym1= rho*sin(phi0)
M= (nu*cos(psi), nu*sin(psi), -1)
nm:  (-2*c*rho*cos(phi0), -2*c*rho*sin(phi0), 1) 


full voutm: -4*(2*c*nu*rho*cos(phi0)*cos(psi) + 2*c*nu*rho*sin(phi0)*sin(psi) + 1)*c*rho*cos(phi0)/(4*c^2*rho^2*cos(phi0)^2 + 4*c^2*rho^2*sin(phi0)^2 + 1) + nu*cos(psi) 

 -4*(2*c*nu*rho*cos(phi0)*cos(psi) + 2*c*nu*rho*sin(phi0)*sin(psi) + 1)*c*rho*sin(phi0)/(4*c^2*rho^2*cos(phi0)^2 + 4*c^2*rho^2*sin(phi0)^2 + 1) + nu*sin(psi) 

 2*(2*c*nu*rho*cos(phi0)*cos(psi) + 2*c*nu*rho*sin(phi0)*sin(psi) + 1)/(4*c^2*rho^2*cos(phi0)^2 + 4*c^2*rho^2*sin(phi0)^2 + 1) - 1 


Taylor-expanded voutm: -4*c^2*nu*rho^2*cos(-2*phi0 + psi) - 4*c^2*nu*rho^2*cos(psi) - 4*c*rho*cos(phi0) + nu*cos(psi) 

 4*c^2*nu*rho^2*sin(-2*phi0 + psi) - 4*c^2*nu*rho^2*sin(psi) - 4*c*rho*sin(phi0) + nu*sin(psi) 

 -8*c^2*rho^2 + 4*c*nu*rho*cos(-phi0 + psi) + 1 


II= (rho*cos(phi0), rho*sin(phi0), c*rho^2)
m:  1/4*(4*c^2*rho^2 - 1)/(8*c^3*rho^

In [16]:
# Now reflect the PHOTON on the primary mirror and check how it is seen in the camera
rxgg,rygg,rzgg,rgg,alpha = var('rxgg,rygg,rzgg,rgg,alpha', domain='real')  

#rx01 = L*theta_c*cos(phi) - rho*cos(phi0)    # keep here vin only to zero and first order 
#ry01 = L*theta_c*sin(phi) - rho*sin(phi0) 


# generate the photon vector to first order only here
vin = vector([vin[0],
              vin[1],
              #nu*theta_c*cos(-phi + psi) + 1/2*nu^2 + 1/2*theta_c^2 - 1
              vin[2].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,1).subs((nu*theta_c*cos(-phi + psi))==0)
             ])

print ('vin (Taylor2)',vin[0],'\n',vin[1],'\n',vin[2])

print ('\nvin=',vin,'\n')

#rx01 = rx.collect(nu).taylor(nu,0,1).trig_reduce().simplify()
#ry01 = ry.collect(nu).taylor(nu,0,1).trig_reduce().simplify()

print ('r2=',r2,'\n')

# Camera image in polar coordinates
rxgg = rgg*cos(alpha)  
rygg = rgg*sin(alpha)
rzgg = c*rgg^2

print ('rz01=',rzgg,'\n\n')

ip = vector([rxgg,rygg,rzgg])     # Now 0th and 1st order 
print ('ip= ',ip,'\n\n')

n=var('n', domain='real')  
# n is the vector normal to the paraboloide n=(-x/(2*f),-y/(2*f),1) = (-2cx,-2cy,1)
n=vector([-2*c*rxgg,-2*c*rygg,1])  # see after Eq. (13)
print('n: ',n,'\n\n')

vout_x,vout_y,vout_z=var('vout_x,vout_y,vout_z', domain='real') 

# reflect the photon vector vin on the mirror at impact point ip   vout=vint-(2(vin·n)/|n|^2)
vout=vin-2*vin.dot_product(n)*n/(n.norm()^2)

vout_x = vout[0]#.collect(c).taylor(c,0,order_expansion)#.simplify()#.trig_reduce().simplify().factor().simplify()
vout_y = vout[1]#.collect(c).taylor(c,0,order_expansion)#.simplify()#.trig_reduce().simplify().factor().simplify()
vout_z = vout[2]#.collect(c).taylor(c,0,order_expansion)#.simplify()#.trig_reduce().simplify().factor().simplify()

# Sage does not understand cos^2+sin^2=1
vout_x = vout_x.factor().subs((4*c^2*rho^2*cos(phi0)^2)==4*c^2*rho^2-4*c^2*rho^2*sin(phi0)^2).simplify().trig_reduce()
vout_y = vout_y.factor().subs((4*c^2*rho^2*cos(phi0)^2)==4*c^2*rho^2-4*c^2*rho^2*sin(phi0)^2).simplify().trig_reduce()
vout_z = vout_z.factor().subs((4*c^2*rho^2*cos(phi0)^2)==4*c^2*rho^2-4*c^2*rho^2*sin(phi0)^2).simplify().trig_reduce()

vout_x = vout_x.collect(c).taylor(c,0,2).trig_reduce().simplify()
vout_y = vout_y.collect(c).taylor(c,0,2).trig_reduce().simplify()
vout_z = vout_z.collect(c).taylor(c,0,2).trig_reduce().simplify()

vout_x = vout_x.collect(nu).taylor(nu,0,1).trig_reduce().simplify()
vout_y = vout_y.collect(nu).taylor(nu,0,1).trig_reduce().simplify()
vout_z = vout_z.collect(nu).taylor(nu,0,1).trig_reduce().simplify()

vout_x = vout_x.collect(theta_c).taylor(theta_c,0,3).trig_reduce().simplify()
vout_y = vout_y.collect(theta_c).taylor(theta_c,0,3).trig_reduce().simplify()
vout_z = vout_z.collect(theta_c).taylor(theta_c,0,3).trig_reduce().simplify()

print('vout (Taylor expanded):',vout_x,'\n\n',vout_y,'\n\n',vout_z,'\n\n')

# remove order 3 and 4 corrections
vout_x = vout_x.subs((2*L*c*nu*theta_c^2*cos(-2*phi + psi))==0)
vout_x = vout_x.subs((2*L*c*nu*theta_c^2*cos(psi))==0)
vout_x = vout_x.subs((-2*c*nu*rho*theta_c*cos(-phi + phi0 + psi))==0)
vout_x = vout_x.subs((-2*c*nu*rho*theta_c*cos(-phi - phi0 + psi))==0)
vout_x = vout_x.subs((-2*c*rho*theta_c^2*cos(phi0))==0)

vout_y = vout_y.subs((-2*L*c*nu*theta_c^2*sin(-2*phi + psi))==0)
vout_y = vout_y.subs((2*L*c*nu*theta_c^2*sin(psi))==0)
vout_y = vout_y.subs((-2*c*nu*rho*theta_c*sin(-phi + phi0 + psi))==0)
vout_y = vout_y.subs((2*c*nu*rho*theta_c*sin(-phi - phi0 + psi))==0)
vout_y = vout_y.subs((-2*c*rho*theta_c^2*sin(phi0))==0)

print('vout (after 3rd order removals):',vout_x,'\n\n',vout_y,'\n\n',vout_z,'\n\n')

# Remove also the second orders by hand (remember that L is large and practically takes away one order): 
#  L^2 * c * nu * theta_c^2  --> order 2
#  L   * c * nu   * theta_c   --> order 2 
#  L   *     nu^2 * theta_c   --> order 2
#  L   * c        * theta_c^2 --> order 2
#  L   *     nu   * theta_c^2 --> order 2
#            nu   * theta_c   --> order 2 
#            nu^2             --> order 2 
#                   theta_c^2 --> order 2
#        c^2                  --> order 2 



#vout_z = vout_z.subs((4*L*c*nu*theta_c*cos(-phi + psi))==0)
#vout_z = vout_z.subs((4*L*c*theta_c^2)==0)
#vout_z = vout_z.subs((-4*c*rho*theta_c*cos(-phi + phi0))==0)
#vout_z = vout_z.subs((-4*c*nu*rho*cos(-phi0 + psi))==0)
vout_z = vout_z.subs((-nu*theta_c*cos(-phi + psi))==0)
#vout_z = vout_z.subs((theta_c^2)==0)
vout_z = vout_z.subs((-1/2*theta_c^2)==0)

vout_x = vout_x.subs((2*L*c*theta_c^3*cos(phi))==0)
vout_y = vout_y.subs((2*L*c*theta_c^3*sin(phi))==0)


vout_x = vout_x.subs((2*c*nu*rgg*theta_c*cos(alpha - phi + psi))==0)
vout_y = vout_y.subs((2*c*nu*rgg*theta_c*sin(alpha - phi + psi))==0)
vout_x = vout_x.subs((2*c*rgg*theta_c^2*cos(alpha))==0)
vout_y = vout_y.subs((2*c*rgg*theta_c^2*sin(alpha))==0)
vout_x = vout_x.subs((2*c*nu*rgg*theta_c*cos(-alpha - phi + psi))==0)
vout_y = vout_y.subs((-2*c*nu*rgg*theta_c*sin(-alpha - phi + psi))==0)

print('vout (after 2nd order removals):',vout_x,'\n\n',vout_y,'\n\n',vout_z,'\n\n')

# produces Eq. (14)
print('vout (after trig_expand):',
      vout_x.trig_expand().trig_expand().simplify_full(),'\n\n',
      vout_y.trig_expand().trig_expand().simplify_full(),'\n\n',
      vout_z.trig_expand().trig_expand().simplify_full(),'\n\n')


vout = vector([vout_x,vout_y,vout_z])

print('Codi latex del vector vout:')
latex(vout)

vin (Taylor2) theta_c*cos(phi) + nu*cos(psi) 
 theta_c*sin(phi) + nu*sin(psi) 
 -1

vin= (theta_c*cos(phi) + nu*cos(psi), theta_c*sin(phi) + nu*sin(psi), -1) 

r2= L^2*theta_c^2 - 2*L*rho*theta_c*cos(-phi + phi0) + rho^2 

rz01= c*rgg^2 


ip=  (rgg*cos(alpha), rgg*sin(alpha), c*rgg^2) 


n:  (-2*c*rgg*cos(alpha), -2*c*rgg*sin(alpha), 1) 


vout (Taylor expanded): -4*c^2*rgg^2*theta_c*cos(-2*alpha + phi) - 4*c^2*nu*rgg^2*cos(-2*alpha + psi) - 4*c^2*rgg^2*theta_c*cos(phi) - 4*c^2*nu*rgg^2*cos(psi) - 4*c*rgg*cos(alpha) + theta_c*cos(phi) + nu*cos(psi) 

 4*c^2*rgg^2*theta_c*sin(-2*alpha + phi) + 4*c^2*nu*rgg^2*sin(-2*alpha + psi) - 4*c^2*rgg^2*theta_c*sin(phi) - 4*c^2*nu*rgg^2*sin(psi) - 4*c*rgg*sin(alpha) + theta_c*sin(phi) + nu*sin(psi) 

 -8*c^2*rgg^2 + 4*c*rgg*theta_c*cos(-alpha + phi) + 4*c*nu*rgg*cos(-alpha + psi) + 1 


vout (after 3rd order removals): -4*c^2*rgg^2*theta_c*cos(-2*alpha + phi) - 4*c^2*nu*rgg^2*cos(-2*alpha + psi) - 4*c^2*rgg^2*theta_c*cos(phi) - 4*c^2*nu*rgg^2*cos(

\left(-4 \, c^{2} \mathit{rgg}^{2} \theta_{c} \cos\left(-2 \, \alpha + \phi\right) - 4 \, c^{2} \nu \mathit{rgg}^{2} \cos\left(-2 \, \alpha + \psi\right) - 4 \, c^{2} \mathit{rgg}^{2} \theta_{c} \cos\left(\phi\right) - 4 \, c^{2} \nu \mathit{rgg}^{2} \cos\left(\psi\right) - 4 \, c \mathit{rgg} \cos\left(\alpha\right) + \theta_{c} \cos\left(\phi\right) + \nu \cos\left(\psi\right),\,4 \, c^{2} \mathit{rgg}^{2} \theta_{c} \sin\left(-2 \, \alpha + \phi\right) + 4 \, c^{2} \nu \mathit{rgg}^{2} \sin\left(-2 \, \alpha + \psi\right) - 4 \, c^{2} \mathit{rgg}^{2} \theta_{c} \sin\left(\phi\right) - 4 \, c^{2} \nu \mathit{rgg}^{2} \sin\left(\psi\right) - 4 \, c \mathit{rgg} \sin\left(\alpha\right) + \theta_{c} \sin\left(\phi\right) + \nu \sin\left(\psi\right),\,-8 \, c^{2} \mathit{rgg}^{2} + 4 \, c \mathit{rgg} \theta_{c} \cos\left(-\alpha + \phi\right) + 4 \, c \nu \mathit{rgg} \cos\left(-\alpha + \psi\right) + 1\right)

In [17]:
sg,fx,fy,fm,fxp0,fyp0,fxppi,fyppi,xg,yg,gx,gy=var('sg,fx,fy,fm,fxp0,fyp0,fxpi,fyppi,xg,yg,gx,gy', domain='real')  
# Now we study the photon's crossing point with the focal plane

#vout = vout.subs(theta_c==0)

print ('ip=',ip,'\n')
print ('vout=',vout,'\n')

Fplaneg = vector([xg,yg,1/(4*c)]) # Pla focal c = 1/(4f)
EqPlaFocg=solve((ip+sg*vout-Fplaneg).list(),[sg,xg,yg],solution_dict=True)[0]  # only one solution possible here
 
fm = EqPlaFocg[sg]#.trig_reduce()
fx = EqPlaFocg[xg]#.trig_reduce().collect(theta_c).taylor(theta_c,0,1).simplify().collect(nu).taylor(nu,0,1).simplify().trig_expand().factor().simplify_full().trig_reduce().simplify()
fy = EqPlaFocg[yg]#.trig_reduce().collect(theta_c).taylor(theta_c,0,1).simplify().collect(nu).taylor(nu,0,1).simplify().trig_expand().factor().simplify_full().trig_reduce().simplify()

#fx = fx.collect(c).taylor(c,0,2).collect(nu).taylor(nu,0,2).collect(theta_c).taylor(theta_c,0,3)#.factor()
#fy = fy.collect(c).taylor(c,0,2).collect(nu).taylor(nu,0,2).collect(theta_c).taylor(theta_c,0,3)#.factor()

fm = fm.collect(c).taylor(c,0,1).collect(theta_c).taylor(theta_c,0,1).collect(nu).taylor(nu,0,1).factor().trig_reduce().simplify()
fx = fx.collect(c).taylor(c,0,1)
fy = fy.collect(c).taylor(c,0,1)

fx = fx.collect(nu).taylor(nu,0,1)
fy = fy.collect(nu).taylor(nu,0,1)

fx = fx.collect(theta_c).taylor(theta_c,0,1).factor().trig_reduce()
fy = fy.collect(theta_c).taylor(theta_c,0,1).factor().trig_reduce()

print(' After Taylor expansions: \n')
print('m: ',fm.simplify(),'\n')
print('x: ',fx.simplify(),'\n')
print('y: ',fy,'\n\n')

fm = fm.subs((nu*cos(-2*alpha + phi + psi))==0)
fm = fm.subs((4*nu*rgg^2*theta_c*cos(-phi + psi))==0)

# remove mixed terms nu*theta_c*c , which are an order of magnitude smaller than c^2*nu or c^2*theta_c
fx = fx.subs((nu*cos(alpha - phi + psi) + 2*nu*cos(-alpha + phi + psi) + nu*cos(-alpha - phi + psi))==0)
fy = fy.subs((nu*sin(alpha - phi + psi) + 2*nu*sin(-alpha + phi + psi) - nu*sin(-alpha - phi + psi))==0)

# general case with coma aberration
print('General case: \n')
print('m: ',fm.trig_expand(),'\n')
print('x: ',fx.trig_expand(),'\n')
print('y: ',fy.trig_expand(),'\n\n')

# relative to first-order camera coordinates
fxrel,fyrel = var('fxrel,fyrel',domain='real')

fxrel = (fx.trig_expand()-cos(phi)*theta_c/(4*c)-nu*cos(psi)/(4*c))*4*c/(rgg*cos(alpha))
fyrel = (fy.trig_expand()-sin(phi)*theta_c/(4*c)-nu*sin(psi)/(4*c))*4*c/(rgg*sin(alpha))
fxrel = fxrel.trig_expand().expand().factor().simplify().expand()
fyrel = fyrel.trig_expand().expand().factor().simplify().expand()
print ('fxrel: ',fxrel,'\n')
print ('fyrel: ',fyrel,'\n\n')

#.subs((cos(alpha))==x/rgg).subs((sin(alpha))==y/rgg)
print('General case: \n')
print('m: ',fm.trig_expand(),'\n')
print('x: ',fx.trig_expand(),'\n')
print('y: ',fy.trig_expand(),'\n\n')


fxp0 = fx.subs(psi==0).subs(phi==0).collect(c).taylor(c,0,1).collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify()
fyp0 = fy.subs(psi==0).subs(phi==0).factor().collect(c).taylor(c,0,1).trig_reduce().simplify()
fxppi = fx.subs(psi==0).subs(phi==pi).collect(c).taylor(c,0,1).collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify()
fyppi = fy.subs(psi==0).subs(phi==pi).factor().collect(c).taylor(c,0,1).trig_reduce().simplify()

fxp0 = fxp0.subs((-8*c*nu*rgg*theta_c*cos(alpha))==0)

print('x (psi,phi=0): ',fxp0.factor().trig_reduce().simplify(),'\n')   # yields Eq. (20)
print('y (psi,phi=0): ',fyp0.trig_reduce().simplify(),'\n')

print('x (psi,phi=pi): ',fxppi.factor().trig_reduce().simplify(),'\n')  # yields Eq. (22)
print('y (psi,phi=pi): ',fyppi.trig_reduce().simplify(),'\n')


fx = fx.trig_expand().trig_expand().factor()
fy = fy.trig_expand().trig_expand().factor()

fx = fx.subs((cos(alpha))==x/rgg).subs((sin(alpha))==y/rgg)
fy = fy.subs((cos(alpha))==x/rgg).subs((sin(alpha))==y/rgg)

fx = fx.subs((rgg^2)==x^2+y^2).trig_reduce()
fy = fy.subs((rgg^2)==x^2+y^2).trig_reduce()

print('Cam_X,Y (alpha replaced)\n')
print('x: ',fx,'\n')      # yields Eq. (16) and (17), if 3*g_x*x^2 + g_x*y^2 = 2*g_x*x^2 + g_x*r^2 is made (g_x = theta_c*cos(phi) + nu*cos(psi))
print('y: ',fy,'\n')      # yields Eq. (16) and (17), if 3*g_y*y^2 + g_y*x^2 = 2*g_y*y^2 + g_y*r^2 is made (g_y = theta_c*sin(phi) + nu*sin(psi))

#fxrel = fxrel.trig_expand().trig_expand().factor()
#fyrel = fyrel.trig_expand().trig_expand().factor()

fxrel = fxrel.subs((cos(alpha))==x/rgg).subs((sin(alpha))==y/rgg)
fyrel = fyrel.subs((cos(alpha))==x/rgg).subs((sin(alpha))==y/rgg)

#fxrel = fxrel.subs((rgg^2)==x^2+y^2).trig_reduce()
#fyrel = fyrel.subs((rgg^2)==x^2+y^2).trig_reduce()

print('Delta(Cam_X,Y)/Cam_X,Y (alpha replaced)\n')
print('xrel: ',fxrel,'\n')     
print('yrel: ',fyrel,'\n')      

fxrel = fxrel.subs((x)==L*theta_c*cos(phi)-rho*cos(phi0))
fxrel = fxrel.subs((y)==L*theta_c*sin(phi)-rho*sin(phi0))
fyrel = fyrel.subs((x)==L*theta_c*cos(phi)-rho*cos(phi0))
fyrel = fyrel.subs((y)==L*theta_c*sin(phi)-rho*sin(phi0))

print('Delta(Cam_X,Y)/Cam_X,Y (x,y replaced)\n')
print('xrel: ',fxrel,'\n')     
print('yrel: ',fyrel,'\n')  

print('Cam_X,Y (alpha replaced, phi=0,psi=0)\n')
print('x: ',fx.subs(phi==0).subs(psi==0).simplify(),'\n')  # yields Eq. (19) if 3*x^2 + y^2 = 2*x^2 + r^2 is made
print('y: ',fy.subs(phi==0).subs(psi==0).simplify(),'\n')  # yields Eq. (19) 

fx = fx.subs((theta_c*sin(phi) + nu*sin(psi))==gy)
fy = fy.subs((theta_c*sin(phi) + nu*sin(psi))==gy)
fx = fx.subs((theta_c*cos(phi) + nu*cos(psi))==gx)
fy = fy.subs((theta_c*cos(phi) + nu*cos(psi))==gx)
fx = fx.subs((nu*cos(psi))==gx-theta_c*cos(phi))
fy = fy.subs((nu*sin(psi))==gy-theta_c*sin(phi))

fx = fx.trig_reduce().simplify()
fy = fy.trig_reduce().simplify()

print('Cam_X,Y (gammax, gammay substituted)\n')
print('x: ',fx,'\n')    
print('y: ',fy,'\n')

fx = fx.subs((x)==L*theta_c*cos(phi)-rho*cos(phi0))
fx = fx.subs((y)==L*theta_c*sin(phi)-rho*sin(phi0))
fy = fy.subs((x)==L*theta_c*cos(phi)-rho*cos(phi0))
fy = fy.subs((y)==L*theta_c*sin(phi)-rho*sin(phi0))

print('Cam_X,Y (x, y substituted)\n')
print('x: ',fx,'\n')    
print('y: ',fy,'\n')

print('Cam_X,Y (test for phi,psi=0)\n')
print('x: ',fx.subs(gy==0).subs(gx==(theta_c+nu)).subs(phi==0),'\n')   # yields Eq. (21), 
print('y: ',fy.subs(gy==0).subs(gx==(theta_c+nu)).subs(phi==0),'\n')   # yields Eq. (21), if sin(phi0)*cos(phi0) = 1/2*sin(2phi0) 

print('Cam_X,Y (test for phi=pi,psi=0)\n')
print('x: ',fx.subs(gy==0).subs(gx==(theta_c-nu)).subs(phi==pi),'\n')   # yields Eq. (23), 
print('y: ',fy.subs(gy==0).subs(gx==(theta_c-nu)).subs(phi==pi),'\n')   # yields Eq. (23), if sin(phi0)*cos(phi0) = 1/2*sin(2phi0) 


print('Cam_X,Y (trig-reduce)\n')
print('x: ',fx.trig_reduce().simplify(),'\n')
print('y: ',fy.trig_reduce().simplify(),'\n')

print('Cam_X,Y (trig-reduce), latex\n')
print('x: ',latex(fx.trig_reduce().simplify()),'\n')
print('y: ',latex(fy.trig_reduce().simplify()),'\n')


print('Cam_X,Y (trig-expand)\n')
print('x: ',fx.trig_expand().simplify(),'\n')
print('y: ',fy.trig_expand().simplify(),'\n')


print('Cam_X,Y UP TO ORDER 2 LATEX\n')
print('x: ',latex(fx.trig_reduce().simplify()),'\n')
print('y: ',latex(fy.trig_reduce().simplify()),'\n')




print('m: ',fm,'\n')
#print('x: ',fx.simplify_full(),'\n')
print('x: ',fx,'\n')
print('y: ',fy,'\n')

print('x: ',latex(fx),'\n')
print('y: ',latex(fy),'\n')


# Aquest resultat és correcte en primer ordre!! 
# Ens esperem que:
# x=f*theta_c*cos(phi) + f*nu*cos(psi) 
# y=f*theta_c*sin(phi) + f*nu*sin(psi) 
# i la resta és petita perquè (nu*theta_c) és petit.



ip= (rgg*cos(alpha), rgg*sin(alpha), c*rgg^2) 

vout= (-4*c^2*rgg^2*theta_c*cos(-2*alpha + phi) - 4*c^2*nu*rgg^2*cos(-2*alpha + psi) - 4*c^2*rgg^2*theta_c*cos(phi) - 4*c^2*nu*rgg^2*cos(psi) - 4*c*rgg*cos(alpha) + theta_c*cos(phi) + nu*cos(psi), 4*c^2*rgg^2*theta_c*sin(-2*alpha + phi) + 4*c^2*nu*rgg^2*sin(-2*alpha + psi) - 4*c^2*rgg^2*theta_c*sin(phi) - 4*c^2*nu*rgg^2*sin(psi) - 4*c*rgg*sin(alpha) + theta_c*sin(phi) + nu*sin(psi), -8*c^2*rgg^2 + 4*c*rgg*theta_c*cos(-alpha + phi) + 4*c*nu*rgg*cos(-alpha + psi) + 1) 

 After Taylor expansions: 

m:  1/4*(4*(4*(nu*cos(-2*alpha + phi + psi) + nu*cos(-phi + psi))*rgg^2*theta_c + rgg^2)*c^2 - 4*(rgg*theta_c*cos(-alpha + phi) + nu*rgg*cos(-alpha + psi))*c + 1)/c 

x:  -1/4*(2*(nu*cos(alpha - phi + psi) + 2*nu*cos(-alpha + phi + psi) + nu*cos(-alpha - phi + psi))*c*rgg*theta_c - 4*(rgg^2*theta_c*(cos(-2*alpha + phi) + 2*cos(phi)) + (nu*cos(-2*alpha + psi) + 2*nu*cos(psi))*rgg^2)*c^2 - theta_c*cos(phi) - nu*cos(psi))/c 

y:  -1/4*(2*(nu*sin(alph

In [18]:
# now check the first obstacle: the secondary mirror! 
# D2 is the distance of the secondary mirror w.r.t. the first one 
# R2 is radius of the secondary mirror
eps2,D2,R2,Lmin2,Lmax2,rx2,ry2,r22,m2=var('eps2,D2,R2,Lmin2,Lmax2,rx2,ry2,r22,m2', domain='real')  

ip2 = vector([x,y,D2])

SS=solve((E+m2*vin-ip2).list(),[x,y,m2],solution_dict=True)[0]   # la primera solució és la positiva. 

print('x1: ',SS[x],'\n')
print('y1: ',SS[y],'\n\n')

rx2 = SS[x].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,0)
ry2 = SS[y].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,0)

print('x2: ',rx2,'\n')
print('y2: ',ry2,'\n\n')

print('m: ',SS[m2].simplify().trig_reduce().simplify(),'\n')

# test if the coordinates are found within or outside of the secondary
r22 = (rx2.collect(nu).taylor(nu,0,1).trig_reduce()^2+ry2.collect(nu).taylor(nu,0,1).trig_reduce()^2).factor().trig_reduce().simplify()
#r2 = r2.subs({L:eps/theta_c**2}).collect(eps).taylor(eps,0,1).subs({eps:L*theta_c**2}).trig_reduce().simplify()
print ('r22=',r22,'\n')

r22 = r22.subs((2*D2^2*nu*theta_c*cos(-phi + psi))==0)  # second order correction 
r22 = r22.subs((D2^2*nu^2)==0)
r22 = r22.subs((D2^2*theta_c^2)==0)

print ('r22=',r22.trig_reduce().simplify(),'\n')

TT=solve((r22-R2**2),[L],solution_dict=True)[1]   # la primera solució és Lmax. 

# Unfortunately, sage math simplifies sin^2+cos^2=1 only with other undesired operations, so we have to tell her "by hand"
Lmax2 = TT[L].simplify().factor().subs({cos(-phi + phi0)^2:1-sin(-phi + phi0)^2}).simplify().subs({cos(-phi + psi)^2:1-sin(-phi + psi)^2}).factor().simplify()
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).trig_reduce().simplify().collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify())
print ('Integration limit Lmax2=',Lmax2,'\n')
# see https://www.wolframalpha.com/input?i=%28-nu%5E2*sin%28-phi+%2B+psi%29%5E2%29-%28%28theta%2Bnu*cos%28phi-psi%29%29%5E2-2*nu*theta*cos%28-phi+%2B+psi%29-nu%5E2-theta%5E2
Lmax2 = Lmax2.subs((-D2^2*nu^2*sin(-phi + psi)^2)==(D2^2*(theta_c+nu*cos(phi-psi))^2-2*D2^2*nu*theta_c*cos(-phi + psi)-D2^2*nu^2-D2^2*theta_c^2))
# see https://www.wolframalpha.com/input?i=cos%28-phi+%2B+p%29*cos%28-phi+%2B+psi%29+-+cos%28-p+%2B+psi%29%29
Lmax2 = Lmax2.subs((D2*nu*cos(-phi + phi0)*cos(-phi + psi) - D2*nu*cos(-phi0 + psi))==(D2*nu*sin(phi0-phi)*sin(phi-psi)))
# see https://www.wolframalpha.com/input?i=1%2F2*rho%5E2*cos%28-2*phi+%2B+2*p%29+%2B+R%5E2+-+1%2F2*rho%5E2+%2B+rho%5E2*sin%28p-phi%29%5E2
Lmax2 = Lmax2.subs((1/2*rho^2*cos(-2*phi + 2*phi0) + R2^2 - 1/2*rho^2)==(R2^2-rho^2*sin(phi0-phi)^2))
print ('Integration limit Lmax2 (substitution)=',Lmax2,'\n')

#Lmax2 = Lmax2.subs((-D2^2*nu^2*sin(-phi+psi)^2)==(-D2^2*nu^2+D2^2*nu^2*cos(-phi+psi)^2))
#Lmax2 = Lmax2.subs((D2^2*nu^2*cos(-phi+psi)^2)==(D2^2*(theta_c+nu*cos(phi-psi))^2-2*D2^2*nu*theta_c*cos(-phi+psi)-D2^2*theta_c^2))
#Lmax2 = Lmax2.subs((-D2^2*nu^2*sin(-phi + psi)^2)==0)
#Lmax2 = Lmax2.subs((2*D2^2*nu*theta_c*cos(-phi + psi))==0)
#Lmax2 = Lmax2.subs((D2^2*nu^2)==0)
#Lmax2 = Lmax2.subs((D2^2*theta_c^2)==0)

print ('Integration limit Lmax2 (substitution)=',Lmax2,'\n')
print ('Integration limit Lmax2 (expansion)=',Lmax2.simplify(),'\n')

print (latex(Lmax2.trig_reduce().simplify()))

# Further Taylor expansion does not help...
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).simplify())

TT=solve((r22-R2**2),[L],solution_dict=True)[0]   # la primera solució és Lmin. 

# Unfortunately, sage math simplifies sin^2+cos^2=1 only with other undesired operations, so we have to tell her "by hand"
Lmin2 = TT[L].simplify().factor().subs({cos(-phi + phi0)^2:1-sin(-phi + phi0)^2}).simplify().subs({cos(-phi + psi)^2:1-sin(-phi + psi)^2}).factor().simplify()
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).trig_reduce().simplify().collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify())
print ('Integration limit Lmin2 (expansion)=',Lmin2,'\n')

Lmin2 = Lmin2.subs((-D2^2*nu^2*sin(-phi + psi)^2)==0)
Lmin2 = Lmin2.subs((2*D2^2*nu*theta_c*cos(-phi + psi))==0)
Lmin2 = Lmin2.subs((D2^2*nu^2)==0)
Lmin2 = Lmin2.subs((D2^2*theta_c^2)==0)
Lmin2 = Lmin2.subs((D2*nu*cos(-phi + phi0)*cos(-phi + psi) - D2*nu*cos(-phi0 + psi))==(D2*nu*sin(phi0-phi)*sin(phi-psi)))

print ('Integration limit Lmin2 (substitution)=',Lmin2,'\n')
print ('Integration limit Lmin2 (expansion)=',Lmin2.trig_reduce().simplify(),'\n')

print (latex(Lmin2.trig_reduce().simplify()))

print('m2: ',SS[m2].simplify().trig_reduce().simplify(),'\n')



x1:  c*nu*rho^2*cos(psi) + 1/2*(2*c*rho^2*cos(phi) - (L*nu^2 + 2*D2 - 2*L)*cos(phi))*theta_c - rho*cos(phi0) - 1/2*(L*nu^3 + 2*D2*nu)*cos(psi) 

y1:  -1/2*L*nu^3*sin(psi) + c*nu*rho^2*sin(psi) - D2*nu*sin(psi) - 1/2*(L*nu^2*sin(phi) - 2*c*rho^2*sin(phi) + 2*D2*sin(phi) - 2*L*sin(phi))*theta_c - rho*sin(phi0) 


x2:  -(D2 - L)*theta_c*cos(phi) - D2*nu*cos(psi) - rho*cos(phi0) 

y2:  -(D2 - L)*theta_c*sin(phi) - D2*nu*sin(psi) - rho*sin(phi0) 


m:  -1/2*L*nu^2 + c*rho^2 - D2 + L 

r22= 2*D2^2*nu*theta_c*cos(-phi + psi) - 2*D2*L*nu*theta_c*cos(-phi + psi) + D2^2*nu^2 + D2^2*theta_c^2 - 2*D2*L*theta_c^2 + L^2*theta_c^2 + 2*D2*rho*theta_c*cos(-phi + phi0) - 2*L*rho*theta_c*cos(-phi + phi0) + 2*D2*nu*rho*cos(-phi0 + psi) + rho^2 

r22= -2*D2*L*nu*theta_c*cos(-phi + psi) - 2*D2*L*theta_c^2 + L^2*theta_c^2 + 2*D2*rho*theta_c*cos(-phi + phi0) - 2*L*rho*theta_c*cos(-phi + phi0) + 2*D2*nu*rho*cos(-phi0 + psi) + rho^2 

Integration limit Lmax2= (D2*nu*cos(-phi + psi) + D2*theta_c + rho*cos(-phi +

In [19]:
# now solve for the condition that the *muon* hits the secondary mirror
rmux2,rmuy2,rmu2,mmu2,rhom2=var('rmux2,rmuy2,rmu2,mmu2,rhom2', domain='real')  

ipmu2 = vector([x,y,D2])

SSS=solve((E+mmu2*M-ipmu2).list(),[x,y,mmu2],solution_dict=True)[0]   # la primera solució és la positiva. 

print('mux1: ',SSS[x],'\n')
print('muy1: ',SSS[y],'\n\n')

rmux2 = SSS[x].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,0)
rmuy2 = SSS[y].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,0)

print('mux2: ',rmux2,'\n')
print('muy2: ',rmuy2,'\n\n')

print('mmu: ',SSS[mmu2].simplify().trig_reduce().simplify(),'\n')

# test if the coordinates are found within or outside of the secondary
rmu2 = (rmux2.collect(nu).taylor(nu,0,1).trig_reduce()^2+rmuy2.collect(nu).taylor(nu,0,1).trig_reduce()^2).factor().trig_reduce().simplify()
#r2 = r2.subs({L:eps/theta_c**2}).collect(eps).taylor(eps,0,1).subs({eps:L*theta_c**2}).trig_reduce().simplify()
print ('rmu2=',rmu2,'\n')

#rmu2 = rmu2.subs((2*D2^2*nu*theta_c*cos(-phi + psi))==0)  # second order correction 
#rmu2 = rmu2.subs((D2^2*nu^2)==0)
#rmu2 = rmu2.subs((D2^2*theta_c^2)==0)

print ('rmu2=',rmu2.trig_reduce().simplify(),'\n')

TTT=solve((rmu2-R2**2),[rho],solution_dict=True)[1]   # la primera solució és Lmax.

rhom2 = TTT[rho].simplify().factor()

print ('Condition for rho (expansion)=',rhom2,'\n')


mux1:  1/2*(2*c*rho^2*cos(psi)*sin(nu) - 2*rho*cos(nu)*cos(phi0) - (2*L*nu*cos(nu) + (L*nu^2 + 2*D2 - 2*L)*sin(nu))*cos(psi))/cos(nu) 

muy1:  1/2*(2*c*rho^2*sin(nu)*sin(psi) - 2*L*nu*cos(nu)*sin(psi) - 2*rho*cos(nu)*sin(phi0) - (L*nu^2*sin(psi) + 2*D2*sin(psi) - 2*L*sin(psi))*sin(nu))/cos(nu) 


mux2:  -D2*nu*cos(psi) - rho*cos(phi0) 

muy2:  -D2*nu*sin(psi) - rho*sin(phi0) 


mmu:  -1/2*(L*nu^2 - 2*c*rho^2 + 2*D2 - 2*L)*sec(nu) 

rmu2= D2^2*nu^2 + 2*D2*nu*rho*cos(-phi0 + psi) + rho^2 

rmu2= D2^2*nu^2 + 2*D2*nu*rho*cos(-phi0 + psi) + rho^2 

Condition for rho (expansion)= -D2*nu*cos(-phi0 + psi) + sqrt(D2^2*nu^2*cos(-phi0 + psi)^2 - D2^2*nu^2 + R2^2) 



In [20]:
# Cherenkov light impacting the camera plane

D3,R3,Lmin3,Lmax3,rx3,ry3,r3,m3,rz=var('D3,R3,Lmin3,Lmax3,rx3,ry3,r3,m3,rz', domain='real')  

print ('rx=',rx,'\n')
print ('ry=',ry,'\n')
rz = r2.subs(c==0)*c
print ('rz=',rz,'\n\n')

print ('voutm=',voutm,'\n\n')

ip = vector([rx,ry,rz])

print ('ip=',ip,'\n\n')

ip3 = vector([x,y,D3])

SS=solve((ip+m3*vout-ip3).list(),[x,y,m3],solution_dict=True)[0]   # la primera solució és la positiva. 

print('x3: ',SS[x],'\n')
print('y3: ',SS[y],'\n\n')

rx3 = SS[x].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,0)
ry3 = SS[y].collect(nu).taylor(nu,0,1).collect(theta_c).taylor(theta_c,0,3).collect(c).taylor(c,0,0)

print('x3: ',rx3,'\n')
print('y3: ',ry3,'\n\n')

print('m: ',SS[m3].simplify().trig_reduce().simplify(),'\n')

# test if the coordinates are found within or outside of the secondary
r3 = (rx3.collect(nu).taylor(nu,0,1).trig_reduce()^2+ry3.collect(nu).taylor(nu,0,1).trig_reduce()^2).factor().trig_reduce().simplify()
#r2 = r2.subs({L:eps/theta_c**2}).collect(eps).taylor(eps,0,1).subs({eps:L*theta_c**2}).trig_reduce().simplify()
print ('r3=',r3,'\n')

#r3 = r3.subs((-D3^2*nu^2*sin(-phi + psi)^2 + 2*D3^2*nu*theta_c*cos(-phi + psi) + D3^2*nu^2)==(D3^2*nu^2*cos(-phi + psi)^2) + 2*D3^2*nu*theta_c*cos(-phi + psi))  # second order correction 
#r3 = r3.subs((D3^2*nu^2)==0)
#r3 = r3.subs((D3^2*theta_c^2)==0)

print ('r3=',r3.trig_reduce().simplify(),'\n')

TT=solve((r3-R3**2),[L],solution_dict=True)[1]   # la primera solució és Lmax. 

# Unfortunately, sage math simplifies sin^2+cos^2=1 only with other undesired operations, so we have to tell her "by hand"
Lmax3 = TT[L].simplify().factor().subs({cos(-phi + phi0)^2:1-sin(-phi + phi0)^2}).simplify().subs({cos(-phi + psi)^2:1-sin(-phi + psi)^2}).factor().simplify()
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).trig_reduce().simplify().collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify())
print ('Integration limit Lmax3=',Lmax3,'\n')

Lmax3 = Lmax3.subs((D3*nu*cos(-phi + phi0)*cos(-phi + psi) - D3*nu*cos(-phi0 + psi))==(D3*nu*sin(phi0-phi)*sin(phi-psi)))
#Lmax3 = Lmax3.subs((2*D3^2*nu*theta_c*cos(-phi + psi))==0)
#Lmax3 = Lmax3.subs((D3^2*nu^2)==0)
#Lmax3 = Lmax3.subs((D3^2*theta_c^2)==0)

print ('Integration limit Lmax3 (expansion)=',Lmax3.simplify(),'\n')

print (latex(Lmax3.trig_reduce().simplify()))

# Further Taylor expansion does not help...
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).simplify())

TT=solve((r3-R3**2),[L],solution_dict=True)[0]   # la primera solució és Lmin. 

# Unfortunately, sage math simplifies sin^2+cos^2=1 only with other undesired operations, so we have to tell her "by hand"
Lmin3 = TT[L].simplify().factor().subs({cos(-phi + phi0)^2:1-sin(-phi + phi0)^2}).simplify().subs({cos(-phi + psi)^2:1-sin(-phi + psi)^2}).factor().simplify()
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).trig_reduce().simplify().collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify())
print ('Integration limit Lmin3 (expansion)=',Lmin3,'\n')

Lmin3 = Lmin3.subs((-D2^2*nu^2*sin(-phi + psi)^2)==0)
Lmin3 = Lmin3.subs((2*D2^2*nu*theta_c*cos(-phi + psi))==0)
Lmin3 = Lmin3.subs((D2^2*nu^2)==0)
Lmin3 = Lmin3.subs((D2^2*theta_c^2)==0)

print ('Integration limit Lmin3 (expansion)=',Lmin3.trig_reduce().simplify(),'\n')

print (latex(Lmin3.trig_reduce().simplify()))
print('m: ',SS[m3].simplify().trig_reduce().simplify(),'\n')



rx= -L^2*c*theta_c^3*cos(phi) + 2*L*c*rho*theta_c^2*cos(phi - phi0)*cos(phi) - L^2*c*nu*theta_c^2*cos(psi) + 2*L*c*nu*rho*theta_c*cos(phi - phi0)*cos(psi) + 1/2*L*theta_c^3*cos(phi) + 1/2*L*nu*theta_c^2*cos(-2*phi + psi) + L*nu*theta_c^2*cos(psi) + L*theta_c*cos(phi) - rho*cos(phi0) 

ry= -L^2*c*theta_c^3*sin(phi) + 2*L*c*rho*theta_c^2*cos(phi - phi0)*sin(phi) - L^2*c*nu*theta_c^2*sin(psi) + 2*L*c*nu*rho*theta_c*cos(phi - phi0)*sin(psi) + 1/2*L*theta_c^3*sin(phi) - 1/2*L*nu*theta_c^2*sin(-2*phi + psi) + L*nu*theta_c^2*sin(psi) + L*theta_c*sin(phi) - rho*sin(phi0) 

rz= (L^2*theta_c^2 - 2*L*rho*theta_c*cos(-phi + phi0) + rho^2)*c 


voutm= (-4*c^2*nu*rho^2*cos(-2*phi0 + psi) - 4*c^2*nu*rho^2*cos(psi) - 4*c*rho*cos(phi0) + nu*cos(psi), 4*c^2*nu*rho^2*sin(-2*phi0 + psi) - 4*c^2*nu*rho^2*sin(psi) - 4*c*rho*sin(phi0) + nu*sin(psi), -8*c^2*rho^2 + 4*c*nu*rho*cos(-phi0 + psi) + 1) 


ip= (-L^2*c*theta_c^3*cos(phi) + 2*L*c*rho*theta_c^2*cos(phi - phi0)*cos(phi) - L^2*c*nu*theta_c^2*cos(psi) + 

In [21]:
# check also for square camera obstacles 
A,Lmins,Lmaxs,Lmaxsxl,Lmaxyl,Lmaxsxr,Lmaxyr=var('A,Lmins,Lmaxs,Lmaxsxl,Lmaxyl,Lmaxsxr,Lmaxyr', domain='real')  
Lx1,Lx2,Ly1,Ly2,ax,bx,ay,by=var('Lx1,Lx2,Ly1,Ly2,ax,bx,ay,by',domain='real')
Lx_min,Lx_max,Ly_min,Ly_max=var('Lx_min,Lx_max,Ly_min,Ly_max',domain='real')
sx,sy=var('sx,sy',domain='real')

print ('rx2=',rx2,'\n')
print ('ry2=',ry2,'\n')

ax = rx2.coefficient(L,1)
bx = rx2.subs(L=0)

ay = ry2.coefficient(L,1)
by = ry2.subs(L=0)

print("ax =", ax)
print("bx =", bx)
print("ay =", ay)
print("by =", by)

Lx1 = solve(ax*L + bx == -A, L)[0].rhs()
Lx2 = solve(ax*L + bx ==  A, L)[0].rhs()

Ly1 = solve(ay*L + by == -A, L)[0].rhs()
Ly2 = solve(ay*L + by ==  A, L)[0].rhs()

print ('\nLx1_min=',Lx1,'\n')
print ('\nLx2_max=',Lx2,'\n')
print ('\nLy1_min=',Ly1,'\n')
print ('\nLy2_max=',Ly2,'\n')

assume(A > 0)
assume(theta_c > 0)
assume(theta_c*cos(phi) != 0)
assume(theta_c*sin(phi) != 0)

# Define new helper variables: 
Lx0,Ly0 = var('Lx0,Ly0',domain='real')
Lx0 = D2 + (D2*nu*cos(psi) + rho*cos(phi0)) / (theta_c*cos(phi))
Ly0 = D2 + (D2*nu*sin(psi) + rho*sin(phi0)) / (theta_c*sin(phi))

def min_to_abs(u, v):
    return (u + v - abs(u - v))/2

def max_to_abs(u, v):
    return (u + v + abs(u - v))/2

Lx_min = Lx0 - A/(theta_c*abs(cos(phi)))
Lx_max = Lx0 + A/(theta_c*abs(cos(phi)))

Ly_min = Ly0 - A/(theta_c*abs(sin(phi)))
Ly_max = Ly0 + A/(theta_c*abs(sin(phi)))
#
#Lx_min = min_to_abs(Lx1, Lx2).simplify_full()
#Lx_max = max_to_abs(Lx1, Lx2).simplify_full()

#Ly_min = min_to_abs(Ly1, Ly2).simplify_full()
#Ly_max = max_to_abs(Ly1, Ly2).simplify_full()

print ('\nLx_min=',Lx_min,'\n')
print ('\nLx_max=',Lx_max,'\n')
print ('\nLy_min=',Ly_min,'\n')
print ('\nLy_max=',Ly_max,'\n')

Lmins = max_to_abs(Lx_min, Ly_min)
Lmaxs = min_to_abs(Lx_max, Ly_max)

#Lmins = max_to_abs(Lx_min, Ly_min).factor().simplify_rational()
#Lmaxs = min_to_abs(Lx_max, Ly_max).factor().simplify_rational()

print ('\nLmins=',Lmins,'\n')
print ('\nLmaxs=',Lmaxs,'\n')


sx = solve(-A < rx2, L) + solve(rx2 < A, L)
sy = solve(-A < ry2, L) + solve(ry2 < A, L)

print("\nsx =", sx,'\n')
print("\nsy =", sy,'\n')


TTxl=solve((rx2==-A),[L],solution_dict=True)[0]   # la primera solució és Lmax. 
Lmaxsxl = TTxl[L].simplify()
TTxr=solve((rx2==A),[L],solution_dict=True)[0]   # la primera solució és Lmax. 
Lmaxsxr = TTxr[L].simplify()
TTyl=solve((ry2==-A),[L],solution_dict=True)[0]   # la primera solució és Lmax. 
Lmaxsyl = TTyl[L].simplify()
TTyr=solve((ry2==A),[L],solution_dict=True)[0]   # la primera solució és Lmax. 
Lmaxsyr = TTyr[L].simplify()

print(Lmaxsxl )
print(Lmaxsxr )
print(Lmaxsyl )
print(Lmaxsyr )

#.factor().subs({cos(-phi + phi0)^2:1-sin(-phi + phi0)^2}).simplify().subs({cos(-phi + psi)^2:1-sin(-phi + psi)^2}).factor().simplify()
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).trig_reduce().simplify().collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify())
print ('Integration limit Lmaxs for x=',Lmaxsx,'\n')
print ('Integration limit Lmaxs for y=',Lmaxsy,'\n')

#Lmin2 = Lmin2.subs((-D2^2*nu^2*sin(-phi + psi)^2)==0)
#Lmin2 = Lmin2.subs((2*D2^2*nu*theta_c*cos(-phi + psi))==0)
#Lmin2 = Lmin2.subs((D2^2*nu^2)==0)
#Lmin2 = Lmin2.subs((D2^2*theta_c^2)==0)
#Lmin2 = Lmin2.subs((D2*nu*cos(-phi + phi0)*cos(-phi + psi) - D2*nu*cos(-phi0 + psi))==(D2*nu*sin(phi0-phi)*sin(phi-psi)))

print ('Integration limit Lmins (substitution)=',Lmins,'\n')
print ('Integration limit Lmins (expansion)=',Lmins.trig_reduce().simplify(),'\n')

print (latex(Lmins.trig_reduce().simplify()))

Xx = D2*theta_c*cos(phi) + D2*nu*cos(psi) + rho*cos(phi0)
Bx = theta_c*cos(phi)

Xy = D2*theta_c*sin(phi) + D2*nu*sin(psi) + rho*sin(phi0)
By = theta_c*sin(phi)

rx2= -(D2 - L)*theta_c*cos(phi) - D2*nu*cos(psi) - rho*cos(phi0) 

ry2= -(D2 - L)*theta_c*sin(phi) - D2*nu*sin(psi) - rho*sin(phi0) 

ax = theta_c*cos(phi)
bx = -D2*theta_c*cos(phi) - D2*nu*cos(psi) - rho*cos(phi0)
ay = theta_c*sin(phi)
by = -D2*theta_c*sin(phi) - D2*nu*sin(psi) - rho*sin(phi0)

Lx1_min= (D2*theta_c*cos(phi) + D2*nu*cos(psi) + rho*cos(phi0) - A)/(theta_c*cos(phi)) 


Lx2_max= (D2*theta_c*cos(phi) + D2*nu*cos(psi) + rho*cos(phi0) + A)/(theta_c*cos(phi)) 


Ly1_min= (D2*theta_c*sin(phi) + D2*nu*sin(psi) + rho*sin(phi0) - A)/(theta_c*sin(phi)) 


Ly2_max= (D2*theta_c*sin(phi) + D2*nu*sin(psi) + rho*sin(phi0) + A)/(theta_c*sin(phi)) 


Lx_min= D2 - A/(theta_c*abs(cos(phi))) + (D2*nu*cos(psi) + rho*cos(phi0))/(theta_c*cos(phi)) 


Lx_max= D2 + A/(theta_c*abs(cos(phi))) + (D2*nu*cos(psi) + rho*cos(phi0))/(theta_c*cos(phi)) 


Ly_min= D2 - A/(theta_c*abs(sin(phi))) + (D2*nu*sin(psi) + rho*sin(phi0))/(theta_c*sin(phi)) 


Ly_max= D2 + A/(theta_c*abs(sin(phi))) + (D2*nu*sin(psi

NameError: name 'Lmaxsx' is not defined

In [48]:
# OLD CODE BY VICTOR FOR TFG
# 
#Es realitza SEGONA APROXIMACIÓ, suposant punt P de impacte amb vin en paraboloide es P=(vin_x,vin_y,(vin_x^2+vin_y^2)/(4*f))
rx = rx.collect(nu).taylor(nu,0,1).trig_reduce()
ry = ry.collect(nu).taylor(nu,0,1).trig_reduce()
mirror1=vector([rx,ry,((rx^2+ry^2)/(4*f)).trig_reduce().simplify()])
print ('impact point on M1: ',mirror1,'\n')

print('Codi latex per al punt P:')
latex(mirror1)

NameError: name 'f' is not defined

In [283]:
n=var('n', domain='real')  
#Coneixent P es coneix el vector normal al paraboloide n=(-P_x/(2*f),-P_y/(2*f),1)
n=vector([-rx/(2*f),-ry/(2*f),1]).trig_reduce()

print('n:',n)

print('Codi latex del vector n:')
latex(n)

n: (-1/2*(L*theta_c*cos(phi) - rho*cos(phi0))/f, -1/2*(L*theta_c*sin(phi) - rho*sin(phi0))/f, 1)
Codi latex del vector n:


\left(-\frac{L \theta_{c} \cos\left(\phi\right) - \rho \cos\left(\phi_{0}\right)}{2 \, f},\,-\frac{L \theta_{c} \sin\left(\phi\right) - \rho \sin\left(\phi_{0}\right)}{2 \, f},\,1\right)

In [284]:
vout_x,vout_y,vout_z,eps3=var('vout_x,vout_y,vout_z,eps3', domain='real')  

#Es por trobar llavors el vector vout amb la relacio vout=vint-(2(vin·n)/|n|^2)
vout=vin-2*vin.dot_product(n)*n/(n.norm()^2)

vout_x = vout[0].collect(nu).taylor(nu,0,1).simplify().trig_reduce().simplify().factor().simplify()
vout_y = vout[1].collect(nu).taylor(nu,0,1).simplify().trig_reduce().simplify().factor().simplify()
vout_z = vout[2].collect(nu).taylor(nu,0,1).simplify().trig_reduce().simplify().factor().simplify()

print('vout:',vout_x,'\n\n',vout_y,'\n\n',vout_z,'\n\n')

vout = vector([vout_x,vout_y,vout_z])

print('Codi latex del vector vout:')
latex(vout)

vout: -(L^2*theta_c^3*cos(phi) + L^2*nu*theta_c^2*cos(-2*phi + psi) - 2*L*f*nu*theta_c^2*cos(-2*phi + psi) - 2*L*f*nu*theta_c^2*cos(psi) + 2*f*nu*rho*theta_c*cos(-phi + phi0 + psi) - 2*L*nu*rho*theta_c*cos(-phi - phi0 + psi) + 2*f*nu*rho*theta_c*cos(-phi - phi0 + psi) - 2*L*rho*theta_c^2*cos(phi0) + 4*L*f*theta_c*cos(phi) - 4*f^2*theta_c*cos(phi) + rho^2*theta_c*cos(-phi + 2*phi0) + nu*rho^2*cos(-2*phi0 + psi) - 4*f^2*nu*cos(psi) - 4*f*rho*cos(phi0))/(L^2*theta_c^2 - 2*L*rho*theta_c*cos(-phi + phi0) + 4*f^2 + rho^2) 

 -(L^2*theta_c^3*sin(phi) - L^2*nu*theta_c^2*sin(-2*phi + psi) + 2*L*f*nu*theta_c^2*sin(-2*phi + psi) - 2*L*f*nu*theta_c^2*sin(psi) + 2*f*nu*rho*theta_c*sin(-phi + phi0 + psi) + 2*L*nu*rho*theta_c*sin(-phi - phi0 + psi) - 2*f*nu*rho*theta_c*sin(-phi - phi0 + psi) - 2*L*rho*theta_c^2*sin(phi0) + 4*L*f*theta_c*sin(phi) - 4*f^2*theta_c*sin(phi) + rho^2*theta_c*sin(-phi + 2*phi0) - nu*rho^2*sin(-2*phi0 + psi) - 4*f^2*nu*sin(psi) - 4*f*rho*sin(phi0))/(L^2*theta_c^2 - 2*L*rho*t

\left(-\frac{L^{2} \theta_{c}^{3} \cos\left(\phi\right) + L^{2} \nu \theta_{c}^{2} \cos\left(-2 \, \phi + \psi\right) - 2 \, L f \nu \theta_{c}^{2} \cos\left(-2 \, \phi + \psi\right) - 2 \, L f \nu \theta_{c}^{2} \cos\left(\psi\right) + 2 \, f \nu \rho \theta_{c} \cos\left(-\phi + \phi_{0} + \psi\right) - 2 \, L \nu \rho \theta_{c} \cos\left(-\phi - \phi_{0} + \psi\right) + 2 \, f \nu \rho \theta_{c} \cos\left(-\phi - \phi_{0} + \psi\right) - 2 \, L \rho \theta_{c}^{2} \cos\left(\phi_{0}\right) + 4 \, L f \theta_{c} \cos\left(\phi\right) - 4 \, f^{2} \theta_{c} \cos\left(\phi\right) + \rho^{2} \theta_{c} \cos\left(-\phi + 2 \, \phi_{0}\right) + \nu \rho^{2} \cos\left(-2 \, \phi_{0} + \psi\right) - 4 \, f^{2} \nu \cos\left(\psi\right) - 4 \, f \rho \cos\left(\phi_{0}\right)}{L^{2} \theta_{c}^{2} - 2 \, L \rho \theta_{c} \cos\left(-\phi + \phi_{0}\right) + 4 \, f^{2} + \rho^{2}},\,-\frac{L^{2} \theta_{c}^{3} \sin\left(\phi\right) - L^{2} \nu \theta_{c}^{2} \sin\left(-2 \, \phi + \psi\rig

In [321]:
m,fx,fy,fm=var('m,fx,fy,fm', domain='real')  
#Ara s'estudia la incidència del foto vout en el pla focal del telescopi

Fplane = vector([x,y,f]) #Pla focal
EqPlaFoc=solve((mirror1+m*vout-Fplane).list(),[m,x,y],solution_dict=True)[0]

fm = EqPlaFoc[m].trig_reduce()
fx = EqPlaFoc[x].trig_reduce().collect(theta_c).taylor(theta_c,0,1).simplify().collect(nu).taylor(nu,0,1).simplify().trig_expand().factor().simplify_full().trig_reduce().simplify()
fy = EqPlaFoc[y].trig_reduce().collect(theta_c).taylor(theta_c,0,1).simplify().collect(nu).taylor(nu,0,1).simplify().trig_expand().factor().simplify_full().trig_reduce().simplify()

print('m: ',fm,'\n')
print('x: ',fx.subs({f:1/eps3}).collect(eps3).taylor(eps3,0,0).subs({eps3:1/f}).trig_reduce().simplify(),'\n')
print('y: ',fy.subs({f:1/eps3}).collect(eps3).taylor(eps3,0,0).subs({eps3:1/f}).trig_reduce().simplify(),'\n')

# Aquest resultat és correcte en primer ordre!! 
# Ens esperem que:
# x=f*theta_c*cos(phi) + f*nu*cos(psi) 
# y=f*theta_c*sin(phi) + f*nu*sin(psi) 
# i la resta és petita perquè (nu*theta_c) és petit.



m:  -1/4*(L^4*theta_c^4 - 4*L^3*rho*theta_c^3*cos(-phi + phi0) + 2*L^2*rho^2*theta_c^2*(cos(-2*phi + 2*phi0) + 2) - 4*L*rho^3*theta_c*cos(-phi + phi0) - 16*f^4 + rho^4)/(L^2*f*nu*theta_c^3*cos(-phi + psi) - 4*f^2*nu*rho*cos(-phi0 + psi) + 4*f^3 - f*rho^2 - (L^2*f - 4*L*f^2 + (L*f*nu*cos(-2*phi + phi0 + psi) + L*f*nu*cos(-phi0 + psi))*rho)*theta_c^2 + (f*nu*rho^2*cos(-phi + psi) + 2*(L*f - 2*f^2)*rho*cos(-phi + phi0) + 4*(L*f^2 - f^3)*nu*cos(-phi + psi))*theta_c) 

x:  nu*rho*theta_c*cos(phi - phi0 + psi) + 1/2*nu*rho*theta_c*cos(-phi + phi0 + psi) + 1/2*nu*rho*theta_c*cos(-phi - phi0 + psi) + f*theta_c*cos(phi) + f*nu*cos(psi) 

y:  nu*rho*theta_c*sin(phi - phi0 + psi) + 1/2*nu*rho*theta_c*sin(-phi + phi0 + psi) - 1/2*nu*rho*theta_c*sin(-phi - phi0 + psi) + f*theta_c*sin(phi) + f*nu*sin(psi) 



In [327]:
# now check the second obstacle: the camera! 
# D3 is the distance of the camera w.r.t. the first mirror !!  
# R3 is radius of the camera
eps4,D3,R3,Lmax3,rx3,ry3,m3=var('eps4,D3,R3,Lmax3,rx3,ry3,m3', domain='real')  

camera_plane = vector([x,y,D3])

SS=solve((mirror1+m3*vout-camera_plane).list(),[x,y,m3],solution_dict=True)[0]   # la primera solució és la negativa. 
rx3 = SS[x].simplify().subs({theta_c:eps4/nu}).collect(eps4).taylor(eps4,0,1).subs({eps4:nu*theta_c}).trig_reduce().simplify()
ry3 = SS[y].simplify().subs({theta_c:eps4/nu}).collect(eps4).taylor(eps4,0,1).subs({eps4:nu*theta_c}).trig_reduce().simplify()
rx3 = rx3.collect(nu).taylor(nu,0,1).trig_reduce().subs({f:1/eps4}).collect(eps4).taylor(eps4,0,0).subs({eps4:1/f}).trig_reduce().simplify()
ry3 = ry3.collect(nu).taylor(nu,0,1).trig_reduce().subs({f:1/eps4}).collect(eps4).taylor(eps4,0,0).subs({eps4:1/f}).trig_reduce().simplify()
print('x: ',rx3,'\n')
print('y: ',ry3,'\n')
print('m: ',SS[m3].simplify().subs({theta_c:eps4/nu}).collect(eps4).taylor(eps4,0,1).subs({eps4:nu*theta_c}).trig_reduce().simplify(),'\n')

# test if the coordinates are found within or outside of the camera
r3 = (rx3.trig_reduce()^2+ry3.trig_reduce()^2).factor().trig_reduce().simplify()
#r2 = r2.subs({L:eps/theta_c**2}).collect(eps).taylor(eps,0,1).subs({eps:L*theta_c**2}).trig_reduce().simplify()
print ('r3=',r3,'\n')

TT=solve((r3-R3**2),[L],solution_dict=True)[0]   # la primera solució és la negativa. 

# Unfortunately, sage math simplifies sin^2+cos^2=1 only with other undesired operations, so we have to tell her "by hand"
Lmax3 = TT[L].simplify().factor().subs({cos(-phi + phi0)^2:1-sin(-phi + phi0)^2}).simplify().subs({cos(-phi + psi)^2:1-sin(-phi + psi)^2}).factor().simplify()
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).trig_reduce().simplify().collect(theta_c).taylor(theta_c,0,1).trig_reduce().simplify())
print ('Integration limit Lmax3 (expansion)=',Lmax3.factor().simplify(),'\n')

# Further Taylor expansion does not help...
#print ('Integration limit Lmax2=',Lmax2.collect(nu).taylor(nu,0,1).simplify())



x:  D3*theta_c*cos(phi) + L*theta_c*cos(phi) + D3*nu*cos(psi) - rho*cos(phi0) 

y:  D3*theta_c*sin(phi) + L*theta_c*sin(phi) + D3*nu*sin(psi) - rho*sin(phi0) 

m:  -1/4*(nu*rho^6*theta_c*cos(-phi + psi) - 4*(2*L*cos(phi - 2*phi0 + psi) + (D3 + L)*cos(-phi + psi))*f*nu*rho^4*theta_c - 2*(L*theta_c*cos(-phi + phi0) + 2*f*theta_c*cos(-phi + phi0))*rho^5 + 16*(D3*L*f^2*theta_c*cos(phi - 2*phi0 + psi) - L*f^3*theta_c*cos(phi - 2*phi0 + psi) - f^4*theta_c*cos(-phi + psi))*nu*rho^2 + 16*((D3 + L)*f^2*theta_c*cos(-phi + phi0) - f^3*theta_c*cos(-phi + phi0))*rho^3 - 64*(D3*L*f^4*theta_c*cos(-phi + psi) - D3*f^5*theta_c*cos(-phi + psi))*nu - 32*(2*D3*L*f^3*theta_c*cos(-phi + phi0) - (2*D3 + L)*f^4*theta_c*cos(-phi + phi0))*rho)/(32*f^4*nu*rho*cos(-phi0 + psi) - 8*f^2*nu*rho^3*cos(-phi0 + psi) - 16*f^5 - f*rho^4 - 8*(f^3*nu^2*cos(-2*phi0 + 2*psi) + f^3*nu^2 - f^3)*rho^2) - 1/4*(16*D3*f^3 - rho^4 + 4*(D3*f - f^2)*rho^2)/(4*f^2*nu*rho*cos(-phi0 + psi) - 4*f^3 + f*rho^2) 

r3= 2*D3^2*nu*theta_c*cos(